# Parametric CAD Bench v2: open Qwen models with execution feedback

Runs Qwen2.5-Coder 3B / 7B / 32B-AWQ on the 100 official `gnucleus-ai/cad-bench@v2` tasks and scores them with each task's own verifier (`gnucleus-freecad-validator==0.4.0` on conda-forge FreeCAD 1.1.0, the leaderboard's runtime). Protocol: verbatim task instruction, greedy decoding, the model returns answer.py; FreeCAD executes it; a Python error, missing FCStd, or a non-PartDesign document is sent back for up to 3 repairs. The reference part and the grader's spec files are used only for final scoring.

**Runtime -> Change runtime type -> A100 GPU** (32B-AWQ needs >= 24 GB), then **Run all**. Use a runtime that is not still training the distillation student. Save the notebook (Cmd/Ctrl+S) at the end.

In [ ]:
!pip -q install vllm uv
import torch
assert torch.cuda.is_available(), 'Select a GPU runtime'
print(torch.cuda.get_device_name(0), round(torch.cuda.mem_get_info()[1] / 2**30), 'GB')

In [ ]:
#@title FreeCAD 1.1.0 + validator 0.4.0 (same packages as the official verifier image)
!curl -Ls https://micro.mamba.pm/api/micromamba/linux-64/latest | tar -xj -C /content bin/micromamba
!/content/bin/micromamba create -q -y -p /content/fcenv -c conda-forge python=3.12 "freecad=1.1.0" "numpy=2.4.6" scipy pydantic
!echo /content/fcenv/lib > $(/content/fcenv/bin/python -c 'import site; print(site.getsitepackages()[0])')/freecad.pth
!/content/fcenv/bin/python -m pip install -q "gnucleus-freecad-validator==0.4.0"
import os
os.environ['FREECAD_LIB'] = '/content/fcenv/lib:/content/fcenv/Mod:/content/fcenv/share/Mod'
!/content/fcenv/bin/python -c "import FreeCAD, importlib.metadata as m; print(FreeCAD.Version()[:3], m.version('gnucleus-freecad-validator'))"

In [ ]:
#@title Official task suite (Harbor) checked against the locally verified file hashes
import base64, hashlib, json
from pathlib import Path
!uvx --from harbor harbor download gnucleus-ai/cad-bench@v2 -o /content/cadbench
TASKS = Path('/content/cadbench/cad-bench')
HASHES = json.loads('''{"freecad-0603c53148/instruction.md": "20c8ceef8ee66c7bb0255f11025403398bf186c77ba984b12b766d9e89b66a12", "freecad-0603c53148/tests/run_scorer.py": "0ecbf3f5844136ffdb0d549a042f4356164c709d53a8a38eb139363cf5523b73", "freecad-0603c53148/tests/grader/spec.json": "b03249e51fc8d0949f594d7e17d3e57e0ca756d7f04993c9a7246d98febd7c1e", "freecad-0603c53148/tests/grader/param_check.py": "ae47ea6c2d2775bba98e151f4dfd74b688b6aaa81b9e851ebd29a669fcc56b12", "freecad-0603c53148/tests/grader/reference.FCStd": "ec8e17b0a6169ad5c954053fa8af02691eecaa2849837c3253af6168dbfbf68a", "freecad-067a9a8726/instruction.md": "066d2cdcc56fe14b952a2a6b8a4ecc4ed71ec325ff0ddad32784b76d5bc78199", "freecad-067a9a8726/tests/run_scorer.py": "0ecbf3f5844136ffdb0d549a042f4356164c709d53a8a38eb139363cf5523b73", "freecad-067a9a8726/tests/grader/spec.json": "9f1b245545281d3a8f20473445d1441700df3fb0e57a7e2b0c84f76275b87533", "freecad-067a9a8726/tests/grader/param_check.py": "26faff6bf3414ca9180f9d19117c0d6b5a76c85e979723c3d8b406c60713d802", "freecad-067a9a8726/tests/grader/reference.FCStd": "3943e1071860f45b4d66a8ec5149c568b4a981c46b192661382b9aed8c76ee15", "freecad-08d2da3057/instruction.md": "8443062584cec3a7b39c14de2d0e6d5f5e24284561f1b083cd05ab4ef0b0a9f7", "freecad-08d2da3057/tests/run_scorer.py": "0ecbf3f5844136ffdb0d549a042f4356164c709d53a8a38eb139363cf5523b73", "freecad-08d2da3057/tests/grader/spec.json": "18c5ef6287c8f3dd4734b685b920ac281b1d56b26dbf13b0b56b69817a3a48a8", "freecad-08d2da3057/tests/grader/param_check.py": "596e4521fedf1b5034fd0f8b62328f415d354101d9d3d568bf12c83b61eedef3", "freecad-08d2da3057/tests/grader/reference.FCStd": "0ba7b432ad29ddbe421e167a693b0a9b6dd158b602aa3934862888b624e10ed5", "freecad-0e9bcbd4f4/instruction.md": "b4b1b347fa3325e1de31a625edab91e8e96f2c1b55cd5d6a9094ccf3d2dc5a63", "freecad-0e9bcbd4f4/tests/run_scorer.py": "0ecbf3f5844136ffdb0d549a042f4356164c709d53a8a38eb139363cf5523b73", "freecad-0e9bcbd4f4/tests/grader/spec.json": "7d1a9a5296336fa2963bb1bf9766b705afd013821ea53a4e1850c42081ddc11b", "freecad-0e9bcbd4f4/tests/grader/param_check.py": "da60e699fe910b16306212c1a929ce4794109035d7f6cae6a6caf896a8c52a5a", "freecad-0e9bcbd4f4/tests/grader/reference.FCStd": "a2f5c8953f8e0dc1d667f495e20e126931e90763a1f126be5ae08c6fb3267a77", "freecad-0f27beedee/instruction.md": "576784d74c05a1532e1e64c4f335235b5a0b0ae5970b9eb98e467a3f9e4b6ed5", "freecad-0f27beedee/tests/run_scorer.py": "0ecbf3f5844136ffdb0d549a042f4356164c709d53a8a38eb139363cf5523b73", "freecad-0f27beedee/tests/grader/spec.json": "707793112c7b466afd13126a299d9a3dc8b6ce0c360fd44c437b6963d8973d37", "freecad-0f27beedee/tests/grader/param_check.py": "c95120bcb53716e735637688b267d1cd0676a7eb1748159f9c9dc6f63c29d689", "freecad-0f27beedee/tests/grader/reference.FCStd": "3bfacc269aa715b986b7bebb43132da129de633a56a1ccf891e24006d7fcdac9", "freecad-0fe9d9e3ce/instruction.md": "399811c4192ebd7f025f70707ddc62d9c3a49f9cd0cb0ed9de1a07f14afae52d", "freecad-0fe9d9e3ce/tests/run_scorer.py": "0ecbf3f5844136ffdb0d549a042f4356164c709d53a8a38eb139363cf5523b73", "freecad-0fe9d9e3ce/tests/grader/spec.json": "164b096fda7f3e82355196086043654f6256492ced5f8beaca8d2550a7ca1e25", "freecad-0fe9d9e3ce/tests/grader/param_check.py": "b28819fa5090595349f5585ba106efc75dd5879553dc58d746462713645e66f6", "freecad-0fe9d9e3ce/tests/grader/reference.FCStd": "02fb412c9c7449474cb5189d8a81972e5487a314e1d7aa6fb8028d0b34ade4b2", "freecad-14da78600e/instruction.md": "3e5f91b27e6a30d696535f237c24d00449d7696004d4615c9a856b82652e8093", "freecad-14da78600e/tests/run_scorer.py": "0ecbf3f5844136ffdb0d549a042f4356164c709d53a8a38eb139363cf5523b73", "freecad-14da78600e/tests/grader/spec.json": "7995a2473b1b02d543be2cc6a5565004e8b47e4eab7e146b2149568d3b190715", "freecad-14da78600e/tests/grader/param_check.py": "82b952498e7cbbdf0b784cbc30af35d0bd91b0661a4e868d17cc6d2eaa219419", "freecad-14da78600e/tests/grader/reference.FCStd": "8fa25662f77fdb6d1edd7e2d3e35822de6583d42cb995827e4647ab4bc3ebab4", "freecad-163672a65f/instruction.md": "a262af75e7349d7a7e7c7d7f6f6c0f6c1d0d19f95f7f02997388615e654243d9", "freecad-163672a65f/tests/run_scorer.py": "0ecbf3f5844136ffdb0d549a042f4356164c709d53a8a38eb139363cf5523b73", "freecad-163672a65f/tests/grader/spec.json": "dc9748e293b23c770dd5d5af7cc0d95ae66d760e7a3c7fbbad28d7e01f4d92a7", "freecad-163672a65f/tests/grader/param_check.py": "1098cd0223bbc43a8f8d20ee4c12d633173fba012ad55d2f2db57cf0c0d1bbde", "freecad-163672a65f/tests/grader/reference.FCStd": "6fc17809efe5c4c550495f466f5a206e306d4bc0d309a3608e21630a596e7617", "freecad-1b9f95801c/instruction.md": "041585f96141019f8f9cb3f7eeb7270b5f919ed168aba265c8072d064669ed98", "freecad-1b9f95801c/tests/run_scorer.py": "0ecbf3f5844136ffdb0d549a042f4356164c709d53a8a38eb139363cf5523b73", "freecad-1b9f95801c/tests/grader/spec.json": "b135fa014ccdc498b8d0497ac57ded8ed2b5661cbc8f68e8d75130d900fd7f67", "freecad-1b9f95801c/tests/grader/param_check.py": "84527d005e5c84c4fcf2c51ce0b6ceea232577a114f5886aeda2942ac72637b8", "freecad-1b9f95801c/tests/grader/reference.FCStd": "075d2656af7169f4704c71571f77c68345f7d8a3ce6a74a05ab471c1dfba7bdc", "freecad-1db55e40f2/instruction.md": "dfb08566be734774e93c21b0b58123222814e8e005d08c1bde541dc049db5126", "freecad-1db55e40f2/tests/run_scorer.py": "0ecbf3f5844136ffdb0d549a042f4356164c709d53a8a38eb139363cf5523b73", "freecad-1db55e40f2/tests/grader/spec.json": "e1885b1122ef09e6ac2e97225e088ee2700a50cc88caa3d91787633cccdf5eda", "freecad-1db55e40f2/tests/grader/param_check.py": "7da18698be1aa780b353deb5689353e9a3ccfc5f0312f64f4f271692af722ede", "freecad-1db55e40f2/tests/grader/reference.FCStd": "d0d9b064dde196821a6f952f720d5bf29f072f1bda321234867f84e51c0c5790", "freecad-21d1517841/instruction.md": "5c684682f30b2105f850d4423f948bdde22450d44c6ced547a070305f3f7fd53", "freecad-21d1517841/tests/run_scorer.py": "0ecbf3f5844136ffdb0d549a042f4356164c709d53a8a38eb139363cf5523b73", "freecad-21d1517841/tests/grader/spec.json": "fbc15fa837d80d5786ae356f9328146f94f471f84e27e4dd939d99c375510d53", "freecad-21d1517841/tests/grader/param_check.py": "ca2f0ebeb9be5096a56061f4a7c6c604ea2cb27fd5db09bc6d4a8e0270027a7d", "freecad-21d1517841/tests/grader/reference.FCStd": "3153399cc5a9e2e8ac9e5c77965e05e40cb897d7089fc444a40faa0cad91bc06", "freecad-264ab71adf/instruction.md": "4ae092f62757e68ac92a1250bc0ae4c56c2298c230f6fd5226d2febf0463c928", "freecad-264ab71adf/tests/run_scorer.py": "0ecbf3f5844136ffdb0d549a042f4356164c709d53a8a38eb139363cf5523b73", "freecad-264ab71adf/tests/grader/spec.json": "e8af72c70c1481d4b7a757996f610b8dea17ccd64f1ba5901ffc49c4c331bfc7", "freecad-264ab71adf/tests/grader/param_check.py": "0542bf519bc66b8de685d9129ec7e9115de7e17ae1154d56d450c2bab264ccae", "freecad-264ab71adf/tests/grader/reference.FCStd": "c6f5bf07628545715c67ef1aea1a7c5922486d59ed10dc6fe6cbc7a35f6363dc", "freecad-28177ca409/instruction.md": "ddc3c818f5eca0e583b7c9bb429d30f453aed1e2b787ae78ecdecf0b5ffc364b", "freecad-28177ca409/tests/run_scorer.py": "0ecbf3f5844136ffdb0d549a042f4356164c709d53a8a38eb139363cf5523b73", "freecad-28177ca409/tests/grader/spec.json": "b5a1329e6636be0d23892cb899839b61b61889523dc1297f4ae72547c2ec2ed0", "freecad-28177ca409/tests/grader/param_check.py": "b92f1c0a613f9963b7dd59db93ebcaf969f0544b30a88f1de351d33031a13fa3", "freecad-28177ca409/tests/grader/reference.FCStd": "4a6f6f06d24f482a6ee1de2c11065f3c07536557a4f2bbf7e74584079b83362d", "freecad-2c186f799d/instruction.md": "5588ecabd371a3ecc4e54708e953e5dd9209ee1f55459f994e19ad2fb15fd17b", "freecad-2c186f799d/tests/run_scorer.py": "0ecbf3f5844136ffdb0d549a042f4356164c709d53a8a38eb139363cf5523b73", "freecad-2c186f799d/tests/grader/spec.json": "2e6b0e36539fce10a25872c3f8fab25bcfd3783eb3c991e10690fb28832d7f99", "freecad-2c186f799d/tests/grader/param_check.py": "87b25caccf7fa6f3bac312900fffd70536e44ae8ccc5d09302340e0c6cbfd9e1", "freecad-2c186f799d/tests/grader/reference.FCStd": "fdc9a32a413d43adbcfa8c85d01099de556c18a1605b6b77e4d10d7f37a7a7e0", "freecad-2fcd25f3ac/instruction.md": "ef22cd32a2c88762289c3533dae6c31fbcebd6ef7a58e2e6101bc0d72f84e643", "freecad-2fcd25f3ac/tests/run_scorer.py": "0ecbf3f5844136ffdb0d549a042f4356164c709d53a8a38eb139363cf5523b73", "freecad-2fcd25f3ac/tests/grader/spec.json": "69d3ebaac13a48cd93f7872183608626dba83114586b034b749930ac934e93a7", "freecad-2fcd25f3ac/tests/grader/param_check.py": "929e398e3fba00346eac411a662e230a41f1d434ff15b11af03cdb4efaedf6c4", "freecad-2fcd25f3ac/tests/grader/reference.FCStd": "300882bd3fa5da317fa70c0ec9c2640f66a312b7d965db32697fce9b960bb2ef", "freecad-333ba9d41c/instruction.md": "304463e7bc2ca43c79f48719f22b36ffbb5e5880086f580541c57f0794bef837", "freecad-333ba9d41c/tests/run_scorer.py": "0ecbf3f5844136ffdb0d549a042f4356164c709d53a8a38eb139363cf5523b73", "freecad-333ba9d41c/tests/grader/spec.json": "5f2f70c6b087eab960abb3abf03ae5e3e4e80593994150a06f3da0e015103d60", "freecad-333ba9d41c/tests/grader/param_check.py": "33bef187d2e5366ad87d4e09bf5dcdb1b717c3c250ebfe05a893cb57ac58da3c", "freecad-333ba9d41c/tests/grader/reference.FCStd": "1d5818bf41241e11bb2bc1e22aa6b3612c6f8c5e94cc9a823d2a2213c5c6df31", "freecad-33e0cf86f9/instruction.md": "0990820613c04be81df6e7e8506914a7b71e6627d971538780c78cdef7ff24b9", "freecad-33e0cf86f9/tests/run_scorer.py": "0ecbf3f5844136ffdb0d549a042f4356164c709d53a8a38eb139363cf5523b73", "freecad-33e0cf86f9/tests/grader/spec.json": "62fad5aff762dfcd9f71f3896184334c1882320e294e9e154b69b8fdce7e5d93", "freecad-33e0cf86f9/tests/grader/param_check.py": "13743c90b508c8d7f82c408174e94e1400b329421b8d18d9c56c28da9e945bd2", "freecad-33e0cf86f9/tests/grader/reference.FCStd": "1e4d4d86de4b5231af284032a2297ce03491a8f2c73ed086aa38c8ba71e546d5", "freecad-3582a53285/instruction.md": "8e347160b08176a8216771665bd0433f8831c1722431d16f47621d555aaa68e5", "freecad-3582a53285/tests/run_scorer.py": "0ecbf3f5844136ffdb0d549a042f4356164c709d53a8a38eb139363cf5523b73", "freecad-3582a53285/tests/grader/spec.json": "38d6d992368747eedbb70ce05288bb0c0c69e61fdb7969d603b1c4a7a0a04ed9", "freecad-3582a53285/tests/grader/param_check.py": "b4f5cd0c600a1f87e3fe0457b7fdea2e42e962297178530735f01be1a26dc030", "freecad-3582a53285/tests/grader/reference.FCStd": "395ca3f4447ffe1d119e0a64da53e4dce9c204392a484cf83672d1823b971091", "freecad-3ba87acac3/instruction.md": "9899cf8c316bef562ac7314d4fe200be45a5fb75152a8f52dfb8f390a661762a", "freecad-3ba87acac3/tests/run_scorer.py": "0ecbf3f5844136ffdb0d549a042f4356164c709d53a8a38eb139363cf5523b73", "freecad-3ba87acac3/tests/grader/spec.json": "f690f636f44593f2379517ad74bd9ae6e3489bf396eb4ac1e02b89020ada3149", "freecad-3ba87acac3/tests/grader/param_check.py": "62ef3993a193b46615805b48fb32ca4ec1820325b0f9aca918ff64e657cd36a3", "freecad-3ba87acac3/tests/grader/reference.FCStd": "549682dac1302c753801eaf32a0e3661a85f4cbb6e3e47e50c2df601e70d39b6", "freecad-3cebd4c4f6/instruction.md": "902988e6d9cf120ba72aef5df1ca46b1c3a1b06bb43d999a4a9d26164ad02d60", "freecad-3cebd4c4f6/tests/run_scorer.py": "0ecbf3f5844136ffdb0d549a042f4356164c709d53a8a38eb139363cf5523b73", "freecad-3cebd4c4f6/tests/grader/spec.json": "a18f7ce88a57b23e952b60c03e3dcb7f91897f857ed31cc254aea271f4d884ec", "freecad-3cebd4c4f6/tests/grader/param_check.py": "01e55d77eae4dae791e5b3b66ea07c077433eef01ac544807f02a5c96fc0173a", "freecad-3cebd4c4f6/tests/grader/reference.FCStd": "2efcbf97863baee0f7ba53460a4bc4babc321622a4cbb82108ef0c0cfed5b919", "freecad-41b24257f7/instruction.md": "e2c0a4d35f90b2c197750578008cd4e42bf44141f674cfda046b576b7988c9fd", "freecad-41b24257f7/tests/run_scorer.py": "0ecbf3f5844136ffdb0d549a042f4356164c709d53a8a38eb139363cf5523b73", "freecad-41b24257f7/tests/grader/spec.json": "16fdc3c75da2d68aabe536eaabf6c793a933326c98956d63e08908c0c883ac09", "freecad-41b24257f7/tests/grader/param_check.py": "0f1e9ddee7d4c6da68049ecb7d7e9e676c2640a0773e10051bc068e83c575245", "freecad-41b24257f7/tests/grader/reference.FCStd": "de6bfe23bf152a44622996de48703ab8fd7c516e90b3119dc6af05d74226fe93", "freecad-4ca1b8519a/instruction.md": "f636c31db2f0d9244fb5d3c6871ccabd3a584ed5fc87e0c3b296247985db242e", "freecad-4ca1b8519a/tests/run_scorer.py": "0ecbf3f5844136ffdb0d549a042f4356164c709d53a8a38eb139363cf5523b73", "freecad-4ca1b8519a/tests/grader/spec.json": "e1ac63c2632330edf87bb1b935b9e326af19a5f92d6fa8cf9f2c50287f1290f6", "freecad-4ca1b8519a/tests/grader/param_check.py": "dce1e5c091aa1c0a8f1996827325ca0b98ae68d7b7fdccf9fb4fd7f5b66e0760", "freecad-4ca1b8519a/tests/grader/reference.FCStd": "83d1bd1e19edf110ae4d2c509246c8628bfdfa330630f377715d996002fd6842", "freecad-52e0552e1d/instruction.md": "a48553c87ea471bf7ab740d58f9aa04ffab6856713ce3241910bc51411ec02d9", "freecad-52e0552e1d/tests/run_scorer.py": "0ecbf3f5844136ffdb0d549a042f4356164c709d53a8a38eb139363cf5523b73", "freecad-52e0552e1d/tests/grader/spec.json": "ac6d251e2192a2f2a0da73a5230864915555fe193b2744b83790763d9516ab34", "freecad-52e0552e1d/tests/grader/param_check.py": "5ffabd7dcb623508d7663ac036310865d5fb9d6c42fd53f4e4cfe307c0acea5e", "freecad-52e0552e1d/tests/grader/reference.FCStd": "fe6674f2545f44c96a816eb22a0351f38c97263ad3b2de2802d99297b59facf5", "freecad-539c2a6856/instruction.md": "3a8e80e9fce3cb7f56f7f7213fc8eb3c41df79fce417fe9f7900882b6e668a6f", "freecad-539c2a6856/tests/run_scorer.py": "0ecbf3f5844136ffdb0d549a042f4356164c709d53a8a38eb139363cf5523b73", "freecad-539c2a6856/tests/grader/spec.json": "bca5f9aa66a355bc2c4ea5c635d42505d20fffa79f62673489005c7f6c17dcec", "freecad-539c2a6856/tests/grader/param_check.py": "6da398231200bc5f4965628839f89ecb96782897f774e307174734de8bd84623", "freecad-539c2a6856/tests/grader/reference.FCStd": "ff21b31812ab8b2e23c8f6c2b00cda0965e67344a4da2b54520d3c426d32f843", "freecad-53b5d25579/instruction.md": "bba5348f0e5b494f480637531cd974beeff7a6f72e3a521853d7c4f938c5ec2f", "freecad-53b5d25579/tests/run_scorer.py": "0ecbf3f5844136ffdb0d549a042f4356164c709d53a8a38eb139363cf5523b73", "freecad-53b5d25579/tests/grader/spec.json": "acad75a9f16799be1be3aa7e5980a83754a6827c7ae756e82515137fbb2f84f7", "freecad-53b5d25579/tests/grader/param_check.py": "6d55d09f315ec228c8cb0bd9dc63aaf47b5bfeb738ef0f4170b0606f367b8c83", "freecad-53b5d25579/tests/grader/reference.FCStd": "39905642b3da1227df84974e65195fa18e929c9d261b43c897855c7d2e7a6b09", "freecad-553fe598d9/instruction.md": "b4bb2d0f9cd2e36e84e5846966d42436a1ae160375294ea641cfe635d471d00d", "freecad-553fe598d9/tests/run_scorer.py": "0ecbf3f5844136ffdb0d549a042f4356164c709d53a8a38eb139363cf5523b73", "freecad-553fe598d9/tests/grader/spec.json": "f9a23b5a563224d6df1f655ec6c055cc398cacd56a63d4de23d158e5cd561bc4", "freecad-553fe598d9/tests/grader/param_check.py": "e3c125cd31a4904a02b39175d113f3cfa494d4653adc87eb2f353e4d610c046e", "freecad-553fe598d9/tests/grader/reference.FCStd": "4f74f7b774efd010a4d2afeab2fd483f52b93cbf2a92d4cf8a8c383e998b3ffc", "freecad-5709ae4446/instruction.md": "00e277e7806bcfea85065670fea980f60216b2f67dc265a8c8f2739c3cc50487", "freecad-5709ae4446/tests/run_scorer.py": "0ecbf3f5844136ffdb0d549a042f4356164c709d53a8a38eb139363cf5523b73", "freecad-5709ae4446/tests/grader/spec.json": "cdd16d0dffc4528bb52d123fd54556d552aced5df710463499a8e3ef7d24ec74", "freecad-5709ae4446/tests/grader/param_check.py": "e2d63111fab66e921b009a18681cda44a4fdadd68a0ea255f601e45f7fb72501", "freecad-5709ae4446/tests/grader/reference.FCStd": "67d791a202f98334b05ef8b14d2227343fa6bcd25180860a032e1891454eb112", "freecad-58a1b82005/instruction.md": "363dc62de317d7f937ad6ff8d605697c02dcbd6fdb2ac04a3e5b219a988d38fd", "freecad-58a1b82005/tests/run_scorer.py": "0ecbf3f5844136ffdb0d549a042f4356164c709d53a8a38eb139363cf5523b73", "freecad-58a1b82005/tests/grader/spec.json": "d06f1de3a94e0610644088afe5703f2c967f566bb569484ffdc15f8e371c599d", "freecad-58a1b82005/tests/grader/param_check.py": "0bd0f649ceba737c57de56c391c52f040fb38cc40018e62a8cffd08e1df00bec", "freecad-58a1b82005/tests/grader/reference.FCStd": "6c02d607669ec12feb8e1fb97c85ae88dbcc27acb5ffc1acf1341dfe9cff6981", "freecad-58fd45a90e/instruction.md": "76ded958d974fe9f8b147b1d3fc49ae761f2e606ea8327486b0b5daa6841143e", "freecad-58fd45a90e/tests/run_scorer.py": "0ecbf3f5844136ffdb0d549a042f4356164c709d53a8a38eb139363cf5523b73", "freecad-58fd45a90e/tests/grader/spec.json": "8c31ab5d0323bba3ecfcf0b0cb863267e7237232490b47176fee389932b543f6", "freecad-58fd45a90e/tests/grader/param_check.py": "a71fcbeb386dc36be33bd37a597094d6bb54a22844a683833b3918c5a3dcdc3b", "freecad-58fd45a90e/tests/grader/reference.FCStd": "5c0d1afac8ff2ea40b15545e7fa5391fb1ae91ed766c234db4ecd6e2b7c6fe5a", "freecad-5c47cdbfc1/instruction.md": "8ccc97a1aedb719e3d8631ad1deb5d2b9bd1a4732a78ff28387224d27a478dd9", "freecad-5c47cdbfc1/tests/run_scorer.py": "0ecbf3f5844136ffdb0d549a042f4356164c709d53a8a38eb139363cf5523b73", "freecad-5c47cdbfc1/tests/grader/spec.json": "a7751d29133ae475e3df5d043e625a14d9806901a3dd2d28e258c94d66587148", "freecad-5c47cdbfc1/tests/grader/param_check.py": "f795648754e3fa4f9be7f4cf81631e4cc614e3ec538b5aa2c5b4e71e70b825aa", "freecad-5c47cdbfc1/tests/grader/reference.FCStd": "e294d111b372652d393c25b40f80a80618beab3a3de6020c33dcdad1b5e8e53c", "freecad-5c60c7a001/instruction.md": "44219df32621cfb6202f91bd1294a474fffd7783b8cfe6c0995924fa5d27c209", "freecad-5c60c7a001/tests/run_scorer.py": "0ecbf3f5844136ffdb0d549a042f4356164c709d53a8a38eb139363cf5523b73", "freecad-5c60c7a001/tests/grader/spec.json": "73b8dc8394dcce020f41ceb7e3c7e1faf2278bf5a1a7c12ac0bd0b9ae58bd6e8", "freecad-5c60c7a001/tests/grader/param_check.py": "6c87307ef9b027d210dbdbda808dce80fb7d2b4452b0f1a988785e82def49872", "freecad-5c60c7a001/tests/grader/reference.FCStd": "c71d369fa5427668b2dfa566a3d11f8f83a70484e58bf89dffcf1a4a10b5925e", "freecad-5c60c7a002/instruction.md": "589ce70e01a2545b1824e45d14bbe2502d93ec4c6421775bcf8d269c136ec1ce", "freecad-5c60c7a002/tests/run_scorer.py": "0ecbf3f5844136ffdb0d549a042f4356164c709d53a8a38eb139363cf5523b73", "freecad-5c60c7a002/tests/grader/spec.json": "a76e2c2b975c0b4e5d5adb0a569965d0a60c34decf08c39e55e0dd3d8ad7251d", "freecad-5c60c7a002/tests/grader/param_check.py": "345276d3b68d0c7071a05ade6e6c16576301f94e795778e57af66826d348edd2", "freecad-5c60c7a002/tests/grader/reference.FCStd": "7bec61237dcab8cf5e3164cba1369c36d82d900eb3461868a680de4c53c0a4c9", "freecad-5c60c7a003/instruction.md": "81a83fc7369744d799f2545bb091205ea6c4747362098066a0efa3ed497c2a52", "freecad-5c60c7a003/tests/run_scorer.py": "0ecbf3f5844136ffdb0d549a042f4356164c709d53a8a38eb139363cf5523b73", "freecad-5c60c7a003/tests/grader/spec.json": "4cff187694c390cb476a1756d66f8b2a0c1d7bef5d7a26835525553b623bef10", "freecad-5c60c7a003/tests/grader/param_check.py": "c743f4af8d40c4c52423886658e5974cde63ed3c68ff3cc2835975ad54206438", "freecad-5c60c7a003/tests/grader/reference.FCStd": "03d98ff14117f7008202825de971976efaec69825e71f7afcd943d241945b28b", "freecad-5c60c7a004/instruction.md": "8a901cf1c8ac4d423a2f2f71c4c0faa89c225db30212163655a9a65928fd17ac", "freecad-5c60c7a004/tests/run_scorer.py": "0ecbf3f5844136ffdb0d549a042f4356164c709d53a8a38eb139363cf5523b73", "freecad-5c60c7a004/tests/grader/spec.json": "9f6f46111001eec9365c3241b9faf8358dc5ed926f8151485b790ee0917c4bd0", "freecad-5c60c7a004/tests/grader/param_check.py": "f3ab25ee9806cbf9c81e5c2424f4a4f3823b280d500e9b5d59ac59b62f0a92c0", "freecad-5c60c7a004/tests/grader/reference.FCStd": "cb09880b99324ec4e3926454262ae4fea2a5c01c1f8360df5cfd0a11f419918e", "freecad-5c60c7a005/instruction.md": "9e709cc507c79e145b5d6de733fa3b5edf1fc23f153305f5b007b807dde3d2b3", "freecad-5c60c7a005/tests/run_scorer.py": "0ecbf3f5844136ffdb0d549a042f4356164c709d53a8a38eb139363cf5523b73", "freecad-5c60c7a005/tests/grader/spec.json": "9ff92b19b0b5723ba8eae08ba48cb0f984ec20b0ca1a1cd0110a2be8e5f2507b", "freecad-5c60c7a005/tests/grader/param_check.py": "7046d7749226df96a8b7f52e2d109dd2c3b0a6110a7aecd24b6824b60a48da06", "freecad-5c60c7a005/tests/grader/reference.FCStd": "0bd37d574fe643be251a3db38a7df36a8847efe6f93dc8b386eef3d55aac6f59", "freecad-6417ae5005/instruction.md": "a31678977935e6ac166546a702bef87c6124f30ab975948ef9ca296516f890aa", "freecad-6417ae5005/tests/run_scorer.py": "0ecbf3f5844136ffdb0d549a042f4356164c709d53a8a38eb139363cf5523b73", "freecad-6417ae5005/tests/grader/spec.json": "7cd2f8feeb07ce96587dff38737a837ced5f64642a1144baea590c5d543064f0", "freecad-6417ae5005/tests/grader/param_check.py": "16d74e6060f3ad30d6a92220978523cf68a8c41215db687721aca2d93c22c3fa", "freecad-6417ae5005/tests/grader/reference.FCStd": "ba80cf02784f750d7c45124394792386295a15b543dce0adb0b6ad0e3126310e", "freecad-641cfcd541/instruction.md": "5eb00b52b096dd6282930cf70f91f49d8b770f3b6060f464e6129ed1eb8126cf", "freecad-641cfcd541/tests/run_scorer.py": "0ecbf3f5844136ffdb0d549a042f4356164c709d53a8a38eb139363cf5523b73", "freecad-641cfcd541/tests/grader/spec.json": "4f33562441e53b6e4c75a49abbdd53fbbe1e66517753cc1ea1629214c54c36bc", "freecad-641cfcd541/tests/grader/param_check.py": "d9a56b8d8fe0a205668dd6692c99fe8525eb8f89a5a9857a1113615d700f32a1", "freecad-641cfcd541/tests/grader/reference.FCStd": "ac6e6a4b3f7fbccd9048c87009d7df2d9b2ced2636e352e82be307e9cba4d8ca", "freecad-65bef2529b/instruction.md": "4478e3ed35f282d7b60be37beaabd5a0ac914e0ec80c563b71ca32279dc57b28", "freecad-65bef2529b/tests/run_scorer.py": "0ecbf3f5844136ffdb0d549a042f4356164c709d53a8a38eb139363cf5523b73", "freecad-65bef2529b/tests/grader/spec.json": "f5e7abfcb41d9995c998481aaa2752170bff7c54a3b3027c5060abe0962b4ab9", "freecad-65bef2529b/tests/grader/param_check.py": "a3a7903443209db2ed37f0c006058c8ec86946fbf566a69a9d1581c70f70a92d", "freecad-65bef2529b/tests/grader/reference.FCStd": "e1bcd5fa76b44cc9c166404c66ccec51e4146289e41843e533101c5567406838", "freecad-669bfb187c/instruction.md": "96459f276222002cedd867ca05256845c7326ff6a10a942e2432801d7282570f", "freecad-669bfb187c/tests/run_scorer.py": "0ecbf3f5844136ffdb0d549a042f4356164c709d53a8a38eb139363cf5523b73", "freecad-669bfb187c/tests/grader/spec.json": "ae181262d27ab3a31981b2cc01a1fb09cf52ae6bcc5d665f0b4ecbb32678bec7", "freecad-669bfb187c/tests/grader/param_check.py": "23123c2909026c7b43c57617f4b50c839572ebeefde4a5d03c9ec003d2398c2a", "freecad-669bfb187c/tests/grader/reference.FCStd": "0d58648dc7c4238609c7d5b98cba84741b0268e90976009bf526d81e3fb12542", "freecad-67b7983b2c/instruction.md": "1f3c829c37c4d4fc0608a8655325ecdeeb1b166a483ecc539af629be9df9dee2", "freecad-67b7983b2c/tests/run_scorer.py": "0ecbf3f5844136ffdb0d549a042f4356164c709d53a8a38eb139363cf5523b73", "freecad-67b7983b2c/tests/grader/spec.json": "bd4fc77f0aa5cd01a16be4ccc891377c7981dbd7e25d97cbdf6bb3c027561451", "freecad-67b7983b2c/tests/grader/param_check.py": "a2c614d413e78bf811fe36d8cc5a552d9f40aa1821d28cde303451838266d78f", "freecad-67b7983b2c/tests/grader/reference.FCStd": "7db4f5ebec9b86fc4b5a04f8f03391254803a20bf10251f3f52a7cafc3d4d63a", "freecad-6d8fcf43ca/instruction.md": "b511d5d37a6e738029310fd5ef8f19b9acf95d9fb8a4eeb9cbbfeeb0a7ee64e5", "freecad-6d8fcf43ca/tests/run_scorer.py": "0ecbf3f5844136ffdb0d549a042f4356164c709d53a8a38eb139363cf5523b73", "freecad-6d8fcf43ca/tests/grader/spec.json": "6f6ca2f853cfdd702857a65602c46b06184a790f57c09ca928c5ebd7d8f1c667", "freecad-6d8fcf43ca/tests/grader/param_check.py": "e05c843891b962ae7453d3f8156e8a26e2e024338dc1aabd9dacb46f6381029d", "freecad-6d8fcf43ca/tests/grader/reference.FCStd": "fc95a6f8ac539ff490eb8825d648ed004e5df72edf8a42d105326e449c808a09", "freecad-732e9c56ac/instruction.md": "1cd61f0e14cffb150c3c5a9f259ff3517af319a5453b91535751c0118e4b32d3", "freecad-732e9c56ac/tests/run_scorer.py": "0ecbf3f5844136ffdb0d549a042f4356164c709d53a8a38eb139363cf5523b73", "freecad-732e9c56ac/tests/grader/spec.json": "16d09d6f144e53e1a6bc080656d488aafbb316ab7cbf8f95f33f1a1f70344a00", "freecad-732e9c56ac/tests/grader/param_check.py": "77d74987913e48d46e79597f2130c729efd0d4ae654332322a3decd25d895162", "freecad-732e9c56ac/tests/grader/reference.FCStd": "487bf3071509cc8d2ca1b9f9e62c94507d2a0d4dd4df6bd768753cad5ac1b7c9", "freecad-740b422c57/instruction.md": "07ba4c432a8367bd48bdf08a4c57507c4810119429585f33313f8c9df3d700e7", "freecad-740b422c57/tests/run_scorer.py": "0ecbf3f5844136ffdb0d549a042f4356164c709d53a8a38eb139363cf5523b73", "freecad-740b422c57/tests/grader/spec.json": "54bf79ab3ae58335bc2434e361cb8ee645aedb8990b06047a492b71a66ebd62a", "freecad-740b422c57/tests/grader/param_check.py": "827418cba99595bb105bae50fb9d9859402e0c2a7a142861277200c533abd730", "freecad-740b422c57/tests/grader/reference.FCStd": "fd40665e287011049ef07230ed2657569b7d9e465bdd86f3963d0ecfef4fbad9", "freecad-7476ec86b6/instruction.md": "d2babe8c8b3a6485995582a0de7c8756fceb44f8e9a70f82c6d4c0464f7c818f", "freecad-7476ec86b6/tests/run_scorer.py": "0ecbf3f5844136ffdb0d549a042f4356164c709d53a8a38eb139363cf5523b73", "freecad-7476ec86b6/tests/grader/spec.json": "5fbab7d4c6ba6c2ba9dbecd535e042f72a82ff05d1256d84c8611d258b9df5c2", "freecad-7476ec86b6/tests/grader/param_check.py": "63b9538328caaf1639b0727a2b0806ab5ae16d2e0069429017d8f29a98fbb5cf", "freecad-7476ec86b6/tests/grader/reference.FCStd": "583e2d233881927ff1ad6e20779a14b64fd3610d54d5a02922cbe4ad18c7e389", "freecad-77178c2112/instruction.md": "552a2e1bad46cc3b84fbb81fab9487957c9aa0fb99fc7c9a4a2e93a4a2f8a6f9", "freecad-77178c2112/tests/run_scorer.py": "0ecbf3f5844136ffdb0d549a042f4356164c709d53a8a38eb139363cf5523b73", "freecad-77178c2112/tests/grader/spec.json": "eb3ee0b808b27f40e0d1b0e8e61d3012124c907ba0f38bd5061c4424c2ea1ea1", "freecad-77178c2112/tests/grader/param_check.py": "c84b5a83e37fd7342f6f1968f4777e7e467838e5d52d76dd347a2cd061a017b8", "freecad-77178c2112/tests/grader/reference.FCStd": "6fa4f5079636b256b8d17c58fb51e22e5f1ca7e32c96d2dd2b844d5e286bdcf8", "freecad-7aa55dda29/instruction.md": "0588ba50bbbdb629ebd0e30a104fa943bbe2c19dbbd84a4cdcd459bc3fe38d78", "freecad-7aa55dda29/tests/run_scorer.py": "0ecbf3f5844136ffdb0d549a042f4356164c709d53a8a38eb139363cf5523b73", "freecad-7aa55dda29/tests/grader/spec.json": "c32eec01bde0853b0f5d6f3e2e86d90648627bc87a9a2df09dfd963fba3ee607", "freecad-7aa55dda29/tests/grader/param_check.py": "8ac76a0fa599cd9a7a46f8c0b0abafda1aac2f1dddaab00fefb7d64ec2afb2d1", "freecad-7aa55dda29/tests/grader/reference.FCStd": "1114b9c36c48ec8fa5d37e502745d38ceb3d974e052d4950390da4f4d58f5dc9", "freecad-7ba017c45b/instruction.md": "cd118b59529f73bee845055c2b8e7bcf7a5944e409d144d064ecae474833e607", "freecad-7ba017c45b/tests/run_scorer.py": "0ecbf3f5844136ffdb0d549a042f4356164c709d53a8a38eb139363cf5523b73", "freecad-7ba017c45b/tests/grader/spec.json": "d011d7972411a81e2f716ef0982055a37a2dda3a37d134971f4b71148d33e854", "freecad-7ba017c45b/tests/grader/param_check.py": "0b5cb15d7d1ff875276e982415e00f429ec059b895d9856f7aa9c87d4bc7b5f1", "freecad-7ba017c45b/tests/grader/reference.FCStd": "e9c4956da2749bc8692376e88ac38039b51a24a891cf059887a31e680a8e6ea7", "freecad-7c683fcc0b/instruction.md": "2500d105f79c22d9e85767d45b6f84b892a3a5256989b34f61a5562cb2221fba", "freecad-7c683fcc0b/tests/run_scorer.py": "0ecbf3f5844136ffdb0d549a042f4356164c709d53a8a38eb139363cf5523b73", "freecad-7c683fcc0b/tests/grader/spec.json": "b02909924d88a567613bcfff373779413a670e8d7fc91cd6c577f52d4dbe1b0a", "freecad-7c683fcc0b/tests/grader/param_check.py": "ef908d77c9c29c8dcd4f25022bb8bc52fc47e62412407559ce3a9ced568338d6", "freecad-7c683fcc0b/tests/grader/reference.FCStd": "6c8acad384cf7f9f1d8837a84d9322561593c0ca438542d216c1b6655f2b0fda", "freecad-7c8d69d649/instruction.md": "fef37f8f2b78d708d9bc331876d2656d3243382bb71509e4605af7133481fef3", "freecad-7c8d69d649/tests/run_scorer.py": "0ecbf3f5844136ffdb0d549a042f4356164c709d53a8a38eb139363cf5523b73", "freecad-7c8d69d649/tests/grader/spec.json": "d3c95648b3d4a675935f718a3614c5e6a05715a2466bfa967246a49575fbb5fb", "freecad-7c8d69d649/tests/grader/param_check.py": "0efad5bc9ea57e0dc66f6f8995ceae8557879f29d381213a616b394a9f8d24b1", "freecad-7c8d69d649/tests/grader/reference.FCStd": "1e7d8cfc88069aa56808d19ec15f5833f20dd58cd55c5f9b24edf4e4d48806ab", "freecad-82c93fd5d6/instruction.md": "dab1b96d57a573f4c65d5239bb1b83e348e347df4573c96cbd323ed7d7551c58", "freecad-82c93fd5d6/tests/run_scorer.py": "0ecbf3f5844136ffdb0d549a042f4356164c709d53a8a38eb139363cf5523b73", "freecad-82c93fd5d6/tests/grader/spec.json": "eb4dbe8942d21fb0901516b2612674f31dda202f4d316676eaede1385eecccf9", "freecad-82c93fd5d6/tests/grader/param_check.py": "0d5a6aae60eebb9853b514638d151e7dc30a0c662775349aab54b96fe2e1b809", "freecad-82c93fd5d6/tests/grader/reference.FCStd": "8fb2b368265ff199409e0aaddb9c95e73b06aad5415d84d56bfba7288a93941a", "freecad-83ca2dab2e/instruction.md": "b78b7f37bd0fd23a6b9a8ee6f998568b1240aac6455b1545561518e25db45d4d", "freecad-83ca2dab2e/tests/run_scorer.py": "0ecbf3f5844136ffdb0d549a042f4356164c709d53a8a38eb139363cf5523b73", "freecad-83ca2dab2e/tests/grader/spec.json": "3c523c4b94a4bbcf88d2ef5b89cb06b7dc781211a03bafe38e940e2fbd10eaf5", "freecad-83ca2dab2e/tests/grader/param_check.py": "3cf14861e8cc6d7f9746fb729da17ee58846f9420f1a21e5f2fb189238d62d55", "freecad-83ca2dab2e/tests/grader/reference.FCStd": "ad0d3ba4ada0e1e094ec01f0a4cc170f8aa131341ba0217663b244f97536e133", "freecad-8450a6402b/instruction.md": "b939137a0e182c819cbc36d4474caf85c6fcf998837ec7f50ddea08bd7c4cca9", "freecad-8450a6402b/tests/run_scorer.py": "0ecbf3f5844136ffdb0d549a042f4356164c709d53a8a38eb139363cf5523b73", "freecad-8450a6402b/tests/grader/spec.json": "09d154929c9da8ac1098a98901981adb3d05a33f6b8607b1842c7751be3f7209", "freecad-8450a6402b/tests/grader/param_check.py": "895f5ad5ba8fe384b4f7fd67342ccd6efd0b3b299c25cd5d78ac4aa151cd6e61", "freecad-8450a6402b/tests/grader/reference.FCStd": "b81c0ba1b6f8ca37c34ebc2a9ab9823f9b5d51797f8e4f6865031b1fa88f4680", "freecad-85145e0d4e/instruction.md": "e9f56bd7752bd57363f6671c1958858438d3c7f07f5be8cbb33016039933f971", "freecad-85145e0d4e/tests/run_scorer.py": "0ecbf3f5844136ffdb0d549a042f4356164c709d53a8a38eb139363cf5523b73", "freecad-85145e0d4e/tests/grader/spec.json": "a5dfde44934f866ff11155289d135cf31d9b2fac021136a95db8bcaf47ad01aa", "freecad-85145e0d4e/tests/grader/param_check.py": "76bec37420d30a7ccc7709e38d15ae58e8e03d8d9ac25e2524268b11aad63329", "freecad-85145e0d4e/tests/grader/reference.FCStd": "e290e3cbaf8c26b492dda128bedfd5750be9c2aa663f972e345e11a09652c1f4", "freecad-893b9137ff/instruction.md": "f8bc0887c2e7e7c0f07518fbb9d8d9e19eed32e065c4a3fb498a972b17321bb6", "freecad-893b9137ff/tests/run_scorer.py": "0ecbf3f5844136ffdb0d549a042f4356164c709d53a8a38eb139363cf5523b73", "freecad-893b9137ff/tests/grader/spec.json": "022d2c5340f2a50171ebe8efd1fa5d702952aa4b7153201245cf6c7c2995a078", "freecad-893b9137ff/tests/grader/param_check.py": "154903589f8a6af6b3511b8ca0a166e2e6144d16088f07cfd352ab231b7ceea5", "freecad-893b9137ff/tests/grader/reference.FCStd": "9e574d4fc2a8c0c8fb6d54ea0437054dcbfba0c13dd0d68cfbd0af2518267bde", "freecad-8acfa92b7f/instruction.md": "44bce5294653a44a7430f7f56b10b8808bf0a5e388dadc3013e92ac61bbf132d", "freecad-8acfa92b7f/tests/run_scorer.py": "0ecbf3f5844136ffdb0d549a042f4356164c709d53a8a38eb139363cf5523b73", "freecad-8acfa92b7f/tests/grader/spec.json": "2f7d020edceff2bc90b22ee582f76329fcef2ee9d60d5b589fb3a22737d9d28c", "freecad-8acfa92b7f/tests/grader/param_check.py": "fbf29bfa3fe90191b71f9dd0a499d7073fda8e06e09511d6a1572c8bdbe8ca33", "freecad-8acfa92b7f/tests/grader/reference.FCStd": "7bce6737e71f57ec5e7e37c53662a2864fdcf44e8f5d99f761019ab28ec4b062", "freecad-8ba92aba6e/instruction.md": "b40f14087a440f54c2c606bdba6d9bcb9e5630676576ec17248d5f157969e372", "freecad-8ba92aba6e/tests/run_scorer.py": "0ecbf3f5844136ffdb0d549a042f4356164c709d53a8a38eb139363cf5523b73", "freecad-8ba92aba6e/tests/grader/spec.json": "4c6f9b4302c7641cfc6d4c8b1eb72eb2f4a1aa8bc5c3f931bba430f8385a0bfd", "freecad-8ba92aba6e/tests/grader/param_check.py": "cb3eed7325eeb30b8c500d570744824ada86000deec41c6607edbe35558388a1", "freecad-8ba92aba6e/tests/grader/reference.FCStd": "95e2ddd40614421f3d5cc15880d4262c3855291a5b17f1ade5e1dd1d70761374", "freecad-8dce996259/instruction.md": "a93c796a96484a991e3eabbebaa2d83d7942476f8955c2e74602de79b9a874e2", "freecad-8dce996259/tests/run_scorer.py": "0ecbf3f5844136ffdb0d549a042f4356164c709d53a8a38eb139363cf5523b73", "freecad-8dce996259/tests/grader/spec.json": "b3e22c6318f34ff862cc3049b4040e428ca464b8b99fe7293d5abbcfc622b9af", "freecad-8dce996259/tests/grader/param_check.py": "ece907af4b7483e4e9c872293b9946673cd707f3ac9e5226a25cc81c4dc58661", "freecad-8dce996259/tests/grader/reference.FCStd": "7fe5e04d3618c6c1b55e33257e27dc0418d32e837b7bc694c416fcd08f4f5e0e", "freecad-9c9ce03e0f/instruction.md": "8f6285bc2413f0b01c34466480d659d30f181c04fe92c5430f6d1207857377be", "freecad-9c9ce03e0f/tests/run_scorer.py": "0ecbf3f5844136ffdb0d549a042f4356164c709d53a8a38eb139363cf5523b73", "freecad-9c9ce03e0f/tests/grader/spec.json": "29e85deb114e6f2260a129429ded398a9b6db86ee4c71de754a233fee95840ba", "freecad-9c9ce03e0f/tests/grader/param_check.py": "fdc07242151d79cdb66d8aa76b4d26b60d33195680751493c0b039fd5a39edd7", "freecad-9c9ce03e0f/tests/grader/reference.FCStd": "0564009a8395fd0989e3fed49ba45052fda1461a0612860e78194272ae4f84e9", "freecad-aa79a250e3/instruction.md": "116c7bc22b6d50cfa40e5820f87fb7f4125e6d8e2c5fabc76594b6f5e1913b92", "freecad-aa79a250e3/tests/run_scorer.py": "0ecbf3f5844136ffdb0d549a042f4356164c709d53a8a38eb139363cf5523b73", "freecad-aa79a250e3/tests/grader/spec.json": "9152f71588de2edeb61e8ca594235825e9f561222329de6d9787dc4c26570f60", "freecad-aa79a250e3/tests/grader/param_check.py": "b914b3a27a55d9aae92d4011902821864cc7e1e79446a4b9f28e8d0506e7f773", "freecad-aa79a250e3/tests/grader/reference.FCStd": "9da2106d8c7d49ad8054ee1cec0b174731895e887abb64eeb9c2780db07a4308", "freecad-aad4a0fbad/instruction.md": "4fad9c083bb766eaa693ca7fc68420fdb5b25c71b33de84b8ad53e5fc7635220", "freecad-aad4a0fbad/tests/run_scorer.py": "0ecbf3f5844136ffdb0d549a042f4356164c709d53a8a38eb139363cf5523b73", "freecad-aad4a0fbad/tests/grader/spec.json": "f5c66a6ef1231610f773c43e19530029c4582e3d2e504e3f982fbf1cef2edd1c", "freecad-aad4a0fbad/tests/grader/param_check.py": "0931f8295f3085d97f439db0fed35097ebcb6a948b39fa6b6670999052406c1e", "freecad-aad4a0fbad/tests/grader/reference.FCStd": "a88d703fc63663e1470467473a4c8b76ce8de1e5cac807a8f88299d2a17b0627", "freecad-ab46933317/instruction.md": "2901cfb75e190b12ce245354ce6c1cf74862f124a2f3e983368e4160e57ecedf", "freecad-ab46933317/tests/run_scorer.py": "0ecbf3f5844136ffdb0d549a042f4356164c709d53a8a38eb139363cf5523b73", "freecad-ab46933317/tests/grader/spec.json": "ec81ca0fa47a4b8af3fb933fd9bff787934c52fea4b8f11aa553545d0cd36e1d", "freecad-ab46933317/tests/grader/param_check.py": "a18c3f7cce41556a955f7db60e08b2b47188cd1ea6e55a34dd3ca0091694ea73", "freecad-ab46933317/tests/grader/reference.FCStd": "14fc36490f986170be83763db06c96423921292bea117030d60c833f8e027e3d", "freecad-ae6efafb3b/instruction.md": "e08c25613d0fb67bb69679d94edf226bce307212e5e421f0b8f8de3ac959d72f", "freecad-ae6efafb3b/tests/run_scorer.py": "0ecbf3f5844136ffdb0d549a042f4356164c709d53a8a38eb139363cf5523b73", "freecad-ae6efafb3b/tests/grader/spec.json": "e438e2aa66fe6b9bbd5c4917522645fae272d7460f1851dcc2b54f49ef54c565", "freecad-ae6efafb3b/tests/grader/param_check.py": "4ae4af791260dcd20d18b74748110dced676ecb33983c9ad9e0de1b864f18c4d", "freecad-ae6efafb3b/tests/grader/reference.FCStd": "eb960a077d60a30b0ea06fa41df6eadd150ff62f6cf22d1446f79cb36140b3b6", "freecad-aff93db55d/instruction.md": "bccc1d01d016d6e8afa8e57ec7075a56b3d11e711587bb421516695a1b5e2510", "freecad-aff93db55d/tests/run_scorer.py": "0ecbf3f5844136ffdb0d549a042f4356164c709d53a8a38eb139363cf5523b73", "freecad-aff93db55d/tests/grader/spec.json": "6f97f85736c4f93f02a626981081787e058a745baf993df00d374d1f3ce490e5", "freecad-aff93db55d/tests/grader/param_check.py": "ecd42fdf69386d7f03bca83b80a5129fb5b1b3abcaec93f270c8cd543dcbeeef", "freecad-aff93db55d/tests/grader/reference.FCStd": "89967477be9e8f10d6414ec46277bf385123bd15e5aff62637e6407dd435e804", "freecad-b0baf87576/instruction.md": "43c9ac2c8b8e495484fb3eb0c4380c2e87d6a56ff62873ca152269b9301c8c03", "freecad-b0baf87576/tests/run_scorer.py": "0ecbf3f5844136ffdb0d549a042f4356164c709d53a8a38eb139363cf5523b73", "freecad-b0baf87576/tests/grader/spec.json": "6ee69f8a951ca087ba2713051d9b5248929e038802b8edbdcba4e2810c6ab4a2", "freecad-b0baf87576/tests/grader/param_check.py": "dd16e3374bc9853f40228f0aa868bd4fc0234d440c9722d869a586193f618582", "freecad-b0baf87576/tests/grader/reference.FCStd": "33753e81bd68181f8faa90b20086d8b3b366153ca8374d5b336378d79f09de4b", "freecad-b2de850e9c/instruction.md": "61db68eb9d4325e2f4f1e4b56db3b6e607de7c3c19cfc9cc9dc40872d54d19a8", "freecad-b2de850e9c/tests/run_scorer.py": "0ecbf3f5844136ffdb0d549a042f4356164c709d53a8a38eb139363cf5523b73", "freecad-b2de850e9c/tests/grader/spec.json": "89e26d9997adacd708575acd1a85515a19963b967cc4e0e0b30d056f34f5d068", "freecad-b2de850e9c/tests/grader/param_check.py": "ec437dad9bf9068468009a4506812a80e1b23a7faafc712819919f2158c00b1b", "freecad-b2de850e9c/tests/grader/reference.FCStd": "59ec8445e7d431cc7dd5cf4e1993514279a16298cc2a571f635761ed7f5d36ef", "freecad-b55517ee4b/instruction.md": "e128113e61e9afc085d6ae5fff748506ad8a206665c1c5eda496a54f54164447", "freecad-b55517ee4b/tests/run_scorer.py": "0ecbf3f5844136ffdb0d549a042f4356164c709d53a8a38eb139363cf5523b73", "freecad-b55517ee4b/tests/grader/spec.json": "f7677142d700e57e9966d37a7c3bc3158664cd75c088e3d2a84b231df09717b8", "freecad-b55517ee4b/tests/grader/param_check.py": "61a42e9753c8b608f3d5671e79d869e67f177544fe691772ca80a4b96b25f72e", "freecad-b55517ee4b/tests/grader/reference.FCStd": "3ffd65aa3523a36448cc1faff1e0ad3522234bb72543a6b22ac47b753495d48a", "freecad-b849a4d94e/instruction.md": "5922aacc3c3fcfe160e6c1f6fe6501d56f965d14b7b98985972ba54fd975a6d5", "freecad-b849a4d94e/tests/run_scorer.py": "0ecbf3f5844136ffdb0d549a042f4356164c709d53a8a38eb139363cf5523b73", "freecad-b849a4d94e/tests/grader/spec.json": "8c7483aff3630ffb707def2ba78e2da8a7437a4ae5f47301b723608b162332d7", "freecad-b849a4d94e/tests/grader/param_check.py": "1f78fdddc57d5328a9d7d9ed9233fdb32160b056339e122179706bbb27d52aa7", "freecad-b849a4d94e/tests/grader/reference.FCStd": "73bfc116b5cabec0e68a5e893e8ac08bba7d1505b78f6f8102749a537daa07d5", "freecad-bee61f8386/instruction.md": "13777813adc73c8a1f0157adfd7501ff18d06e4d3fb182e07dff788ba2f017e7", "freecad-bee61f8386/tests/run_scorer.py": "0ecbf3f5844136ffdb0d549a042f4356164c709d53a8a38eb139363cf5523b73", "freecad-bee61f8386/tests/grader/spec.json": "39aade1bec6493fff94f535b40f3bfc3e1f45d7fdd7c9da35d9f8f4c342e5be4", "freecad-bee61f8386/tests/grader/param_check.py": "d8288943bcf32a3503f563eb962bf35c60fb97ab65e9ad5b006dbf26db591e41", "freecad-bee61f8386/tests/grader/reference.FCStd": "2632634483a39ea2944bc40ef8c85e1ae90e130e39ba558ec84c4e6e00d843ee", "freecad-bf26b04486/instruction.md": "a09ef39a217b6590c14104755222522d5b796bfc630c065fc87143a5023e8103", "freecad-bf26b04486/tests/run_scorer.py": "0ecbf3f5844136ffdb0d549a042f4356164c709d53a8a38eb139363cf5523b73", "freecad-bf26b04486/tests/grader/spec.json": "0bebcd29a799ac6acacefb761b24924f1c076997698a2a67592a261fd06175a4", "freecad-bf26b04486/tests/grader/param_check.py": "e3178bdd2d57ba9d43ee7aaa31711a519e08de7d2c7bf6531ecc194ef7651793", "freecad-bf26b04486/tests/grader/reference.FCStd": "339418cedc23d58f7acc30cc9b1476f8ee2cd1d1174c27220a95e9c43cbc4e1f", "freecad-c4bad6c5af/instruction.md": "35e566dad0023aa4bcc6e0c2be3da920611ceeedef047cebab424152b363088e", "freecad-c4bad6c5af/tests/run_scorer.py": "0ecbf3f5844136ffdb0d549a042f4356164c709d53a8a38eb139363cf5523b73", "freecad-c4bad6c5af/tests/grader/spec.json": "ef0512371f4405554720c6a1e220d0996660f107f78c6dcadba61b21ae06ab11", "freecad-c4bad6c5af/tests/grader/param_check.py": "2b3dbc1a990625e548f3659202db6ea2f147bb09e56597adfd744cc591664b03", "freecad-c4bad6c5af/tests/grader/reference.FCStd": "63c437ec393041f4bf8911b85fb61b456c57c7055c3b6b8ee86e508a62a6f934", "freecad-c5551173e6/instruction.md": "559f85ed3437b6c38e01b2ad9a982a83a0b943e26359b9417c74e349c95f8ec4", "freecad-c5551173e6/tests/run_scorer.py": "0ecbf3f5844136ffdb0d549a042f4356164c709d53a8a38eb139363cf5523b73", "freecad-c5551173e6/tests/grader/spec.json": "a0cac0dcf75f13c09d767cc0699ca871f267ded06c5a111ce9c6c1c64b34e073", "freecad-c5551173e6/tests/grader/param_check.py": "04e90a379e85241e17dbc29e7ffe0dd1de0d84fed4e54a731f723f2032328ee0", "freecad-c5551173e6/tests/grader/reference.FCStd": "b6070b5cf1d498db5c65565cd19cb25aa88acff0f9e463431f326f6d183d6d2b", "freecad-c7364c0bbd/instruction.md": "c2ee1a3e76bf04e4ac1ab6f75df056d8a5bda743b1bf1b8407177cc10dde18b4", "freecad-c7364c0bbd/tests/run_scorer.py": "0ecbf3f5844136ffdb0d549a042f4356164c709d53a8a38eb139363cf5523b73", "freecad-c7364c0bbd/tests/grader/spec.json": "2d7923c0835c712f67d1083a0f1c7a98fd55d5e15e05d5c7614f1a7e62d54cd3", "freecad-c7364c0bbd/tests/grader/param_check.py": "ebaf002e201d17b5af794b6336cf32331e6fe21ad05a97367285549dbf8997e2", "freecad-c7364c0bbd/tests/grader/reference.FCStd": "d42c41f442f174d933b4f8af87a9805e8e46bcf83ba4611d05f65de40b3097f3", "freecad-c7ccc5660f/instruction.md": "3801630d4ff3fb2f7429225a00f5b3e2467d8ae24796a033e874de41963c6a5e", "freecad-c7ccc5660f/tests/run_scorer.py": "0ecbf3f5844136ffdb0d549a042f4356164c709d53a8a38eb139363cf5523b73", "freecad-c7ccc5660f/tests/grader/spec.json": "5876f9092454a6bd8844eb5e2ff35fe78a5e7a5d999327ec8ce2eed9065cf678", "freecad-c7ccc5660f/tests/grader/param_check.py": "e33d8e17b50b972d69e4bad62ee0eafc8dd569c820c25aa0878d28959c129028", "freecad-c7ccc5660f/tests/grader/reference.FCStd": "5f6618b6c61c9df7df0a376aa4337a25f83b50ea0be1003829b36dcd26bbc5ad", "freecad-cb6f0c2fb5/instruction.md": "9d06388a6af13fc703a6cb8da5adf2734e2d90ef84a5cd1636c4a177617d9975", "freecad-cb6f0c2fb5/tests/run_scorer.py": "0ecbf3f5844136ffdb0d549a042f4356164c709d53a8a38eb139363cf5523b73", "freecad-cb6f0c2fb5/tests/grader/spec.json": "74d435c13694b04d466cfadf691a39ca9cef97f6bdb680cddbd5978b018e022d", "freecad-cb6f0c2fb5/tests/grader/param_check.py": "b0d1eecad0896d009dceefdf10dcfd8b1f91a12af06105bd1b9bffc430e8bd1b", "freecad-cb6f0c2fb5/tests/grader/reference.FCStd": "bdd15a873c6cb53445144a3ae0f72acc28675f8eebee30b412f4540f6d5f5e5b", "freecad-cda06f9dbf/instruction.md": "cd8b64c5ddae935a293974a3a9bb28a979ce5d6946159d23cc5f9df5677b445c", "freecad-cda06f9dbf/tests/run_scorer.py": "0ecbf3f5844136ffdb0d549a042f4356164c709d53a8a38eb139363cf5523b73", "freecad-cda06f9dbf/tests/grader/spec.json": "affba26116b976882f8d6a80a3075e8a2bf7e35244fe56548a493a2062239b18", "freecad-cda06f9dbf/tests/grader/param_check.py": "d206a03257f444053c4e85b1b3c0c94412ca2a16b9d998e81eb83d6f6b655dea", "freecad-cda06f9dbf/tests/grader/reference.FCStd": "5976edeb5cd62aa246efe460574803a8dca0af77dcaa556d28100202d8e5d35d", "freecad-d733bfa285/instruction.md": "0eadba9ab29132a4a60f95a356acf5c9dc01a66cd77d8f5f58931ae9a814858c", "freecad-d733bfa285/tests/run_scorer.py": "0ecbf3f5844136ffdb0d549a042f4356164c709d53a8a38eb139363cf5523b73", "freecad-d733bfa285/tests/grader/spec.json": "98db4f6edad539963bb29600dc16f1f256f91007cfa6910b9ef09759332206b4", "freecad-d733bfa285/tests/grader/param_check.py": "fd77c7b734b9ded3465104417aaa33796b6630e846954fb96abb306b295e4b78", "freecad-d733bfa285/tests/grader/reference.FCStd": "78eeda2fd8fe929e922f747c77d84af13c0f526675be7e9fe02c4582b6aa9d9a", "freecad-d7e1bdf8be/instruction.md": "f990c9445c426112ffbfbbb3ac67c95d08402fe91ada71f9a76012557a071dc2", "freecad-d7e1bdf8be/tests/run_scorer.py": "0ecbf3f5844136ffdb0d549a042f4356164c709d53a8a38eb139363cf5523b73", "freecad-d7e1bdf8be/tests/grader/spec.json": "4c6803c46d01a8331278dd6f41fce98dd1d6973f9a124ad44925909f32fd5ef8", "freecad-d7e1bdf8be/tests/grader/param_check.py": "1a0da66e9176774b1cf950942e6ca179a9f844f6261cc8ef84e9aad493800310", "freecad-d7e1bdf8be/tests/grader/reference.FCStd": "c2f339a1d85c29401c677e2aa600aaf74801b476b3839aa2208f75605e097f91", "freecad-d831ec007c/instruction.md": "9433b5ae10cd089a6deff0e4ef7e582865bd2add1e2d9ededcaf7c82f04f3a39", "freecad-d831ec007c/tests/run_scorer.py": "0ecbf3f5844136ffdb0d549a042f4356164c709d53a8a38eb139363cf5523b73", "freecad-d831ec007c/tests/grader/spec.json": "d325ad4d0e525711829981e9bb8eee08132dbd449c313c31dee17d68aab27256", "freecad-d831ec007c/tests/grader/param_check.py": "484c4e9b27e15c7efee2cf3f08bbdfd39aada0a463405b08a701f45b599d423e", "freecad-d831ec007c/tests/grader/reference.FCStd": "9a71a8e80611296909ed688774bdd229f1b60d6ba91e2256d9de7c3e6f64d723", "freecad-da772a2e0b/instruction.md": "9b08ba797886001ecd1ae994a3ff2ebf5796b6514f1c8ceea9619d87eea32e99", "freecad-da772a2e0b/tests/run_scorer.py": "0ecbf3f5844136ffdb0d549a042f4356164c709d53a8a38eb139363cf5523b73", "freecad-da772a2e0b/tests/grader/spec.json": "4c673310ac4616c72eaaada00bd033e5d3f1f097d7c05afd67019b8cb22b1c02", "freecad-da772a2e0b/tests/grader/param_check.py": "159bbb4c55fca49168ff6852a06641c769db96ed09e7ce01aa6b64db857d893d", "freecad-da772a2e0b/tests/grader/reference.FCStd": "fbfd981e7b16dbdd484ae759f29359c541d0a60aa61e7444dfbcb43d48d0cde7", "freecad-db1ef53f08/instruction.md": "4bb1bee89b06860ffff8324ab5f8277abd4b8f66bf78924b814ce16f31b400ca", "freecad-db1ef53f08/tests/run_scorer.py": "0ecbf3f5844136ffdb0d549a042f4356164c709d53a8a38eb139363cf5523b73", "freecad-db1ef53f08/tests/grader/spec.json": "3423dab84f5bf3fdce93e555952e52ea583f648a9d3198ebe942e5a9b51ddd14", "freecad-db1ef53f08/tests/grader/param_check.py": "4c39835a2ad2cfd22e9c7d6bc1906273a7bb94ef0b125ed38af51e1cf5de1022", "freecad-db1ef53f08/tests/grader/reference.FCStd": "de7f22ab80805d02039de0fa492d618f017f4426f970d2c7470c95873546d6b4", "freecad-db4e2a4bcf/instruction.md": "579bb4f7a126e55eeeb2b4c206a4cbe0f49ef8beff5ca29b55b90aafa4e37938", "freecad-db4e2a4bcf/tests/run_scorer.py": "0ecbf3f5844136ffdb0d549a042f4356164c709d53a8a38eb139363cf5523b73", "freecad-db4e2a4bcf/tests/grader/spec.json": "44c4de2e844a9b618ec7048283a96edcb4173cf60812b6acca5433bf5b23be27", "freecad-db4e2a4bcf/tests/grader/param_check.py": "7ea42ceb9f32666842dd1cc67f0d80381f4fce1062c6e57f218fe61889794ffd", "freecad-db4e2a4bcf/tests/grader/reference.FCStd": "4adbdce00713564f80ebd0b8bb3283dbb38bb55470f8c8e6ce7cb3f7898e16ff", "freecad-dbf8356a78/instruction.md": "711d55162283395c9facc7e797ed7f2bc382358b22468df89f56f29b5e5e64fa", "freecad-dbf8356a78/tests/run_scorer.py": "0ecbf3f5844136ffdb0d549a042f4356164c709d53a8a38eb139363cf5523b73", "freecad-dbf8356a78/tests/grader/spec.json": "ea547ad6296a76e399848ba73298c0fc45fa98a7fbfa584cca1c0e5cb5487b7c", "freecad-dbf8356a78/tests/grader/param_check.py": "e2ed20bc84951c4e6fe3cc3ee480b5dad9376fabbf323da8b93f50a517e9f604", "freecad-dbf8356a78/tests/grader/reference.FCStd": "f6df993f35dc1d44028e1ff1b89dceb38808a5444c56316478a7f873ea6cead4", "freecad-dc935d877e/instruction.md": "0da15314ed94f4641172b887bd1d9c6e82984bc01c46c9391637545a5e8171ff", "freecad-dc935d877e/tests/run_scorer.py": "0ecbf3f5844136ffdb0d549a042f4356164c709d53a8a38eb139363cf5523b73", "freecad-dc935d877e/tests/grader/spec.json": "7de07e94d162c255c19fd99476d7ca70251dd122a6803d58a7e2c1ee04ce93f4", "freecad-dc935d877e/tests/grader/param_check.py": "61918db01c4b894b94d43180b04113cdb996280e24698280fcddfc4e464bfbf8", "freecad-dc935d877e/tests/grader/reference.FCStd": "ba89c313a736a2297ae01021534a2ca3ef85a5a2f41d1385ca3b9ecd84bdbf8a", "freecad-e0b89a1ecc/instruction.md": "0afe71a524dc6936426c73fd6492e8afbe17d6d0a9bed12b393286bb2a912641", "freecad-e0b89a1ecc/tests/run_scorer.py": "0ecbf3f5844136ffdb0d549a042f4356164c709d53a8a38eb139363cf5523b73", "freecad-e0b89a1ecc/tests/grader/spec.json": "9c5986c7becfa30004d49dcfbb9b58be2426da99e6b6f5a44ee1a04b89c48364", "freecad-e0b89a1ecc/tests/grader/param_check.py": "177d9a80e98d818d2316ef349e4c5f1a40f32cbe80836c7c5949d506b890396d", "freecad-e0b89a1ecc/tests/grader/reference.FCStd": "814db15edaeab7f192c8a480117b3bfc0770af9a5619e1dc1190f91556428a58", "freecad-e20918a733/instruction.md": "16a2e181a401ced9cd49404c5afa443bd62c73dc19bbf217c10975bb94f9c5bd", "freecad-e20918a733/tests/run_scorer.py": "0ecbf3f5844136ffdb0d549a042f4356164c709d53a8a38eb139363cf5523b73", "freecad-e20918a733/tests/grader/spec.json": "0a4489f773c97037505cf0190de899ca463c087843af1c8113ccb5fa1225a27f", "freecad-e20918a733/tests/grader/param_check.py": "7e5f7035071db3515729e1079f0045126361164c72ce6e65fe45c8b46e3eadf4", "freecad-e20918a733/tests/grader/reference.FCStd": "75be4187b7fb7f13e4b5a3e82aa03943d2a4c8416b89fff2017a15ed5f7e6a9e", "freecad-e4cf76d978/instruction.md": "d79ea007e3222a8c7b34c57f82c05580fb3cd04f9d10076dd10750e9535c1f98", "freecad-e4cf76d978/tests/run_scorer.py": "0ecbf3f5844136ffdb0d549a042f4356164c709d53a8a38eb139363cf5523b73", "freecad-e4cf76d978/tests/grader/spec.json": "687fe2547dde04060a19cf04c42e74d098c99cb0652b91e1692baa205cfce428", "freecad-e4cf76d978/tests/grader/param_check.py": "79e8e59f53e4d942bb86f55a1e2c2a3506a18bdb5214e934ecd601468d2ede5e", "freecad-e4cf76d978/tests/grader/reference.FCStd": "ee969e510b479483db6c78ad28a55937fd499ec707ea29197bfb6f67fcfe483c", "freecad-e50de66084/instruction.md": "e44b459d6c6b274eb36ef6fb752e364c0a53623b6e5f9af3734d09c628a9f2d1", "freecad-e50de66084/tests/run_scorer.py": "0ecbf3f5844136ffdb0d549a042f4356164c709d53a8a38eb139363cf5523b73", "freecad-e50de66084/tests/grader/spec.json": "800d62e1499a93a445564e01e1a748383d444ad974bb8f8e20bde24fbf15df4b", "freecad-e50de66084/tests/grader/param_check.py": "4d3d9e79951ae12c08518a395c692144a78d5ccbef47c6d70a3eab080ae546eb", "freecad-e50de66084/tests/grader/reference.FCStd": "2ca7b037fa71ac59839d52254bb2ecdbc14d3b134adc0a6ace9bb4bd74cb3b76", "freecad-e74ef5c001/instruction.md": "d6bd8bbc18f0541ebf6851ded2821c24d17f8c42dc9a283293381d8667a57f86", "freecad-e74ef5c001/tests/run_scorer.py": "0ecbf3f5844136ffdb0d549a042f4356164c709d53a8a38eb139363cf5523b73", "freecad-e74ef5c001/tests/grader/spec.json": "830ba20f8599eebdfc39990348a9bcdf18d88f911d1936c67a567f6b0199cc79", "freecad-e74ef5c001/tests/grader/param_check.py": "1999660cad18539a5e11a3595333524caf5f841c0b3ce8aeb418f7f3472c847f", "freecad-e74ef5c001/tests/grader/reference.FCStd": "1bbfb862ef711b83eafc89180aee7283a7a6d8e17612c4c0d4c74ce8e4849ec6", "freecad-e74ef5c002/instruction.md": "5307ce4bbe61d80c18c56cda7b30fe293ad7eb735c8150c4b5e7a6baac6d20d8", "freecad-e74ef5c002/tests/run_scorer.py": "0ecbf3f5844136ffdb0d549a042f4356164c709d53a8a38eb139363cf5523b73", "freecad-e74ef5c002/tests/grader/spec.json": "ec2ce289813a439a6f012bf5ee60d7646e406531af1a3003f53faea2acfdc14d", "freecad-e74ef5c002/tests/grader/param_check.py": "fe05429fc36d38e624b9e330b0ce21f6ee7cf7b35e944c9e4323b0c2a3239576", "freecad-e74ef5c002/tests/grader/reference.FCStd": "5135bd983ee353bd044602138575da040b74264b4e94fa5949ed507a05b44da3", "freecad-e74ef5c003/instruction.md": "76d6643072f576c1ee8a3513e81d000b3b8615c66582ae0cca1fdf1a9bd6b741", "freecad-e74ef5c003/tests/run_scorer.py": "0ecbf3f5844136ffdb0d549a042f4356164c709d53a8a38eb139363cf5523b73", "freecad-e74ef5c003/tests/grader/spec.json": "4f1d78f152dc869b0a68878c0c00e0a0264e198058a8a90ac5e7fc6b0a5af3f9", "freecad-e74ef5c003/tests/grader/param_check.py": "fe4859532c27bd27a6eebf918ef5403821192fa8fe3c49bfe2bccb56436bec8f", "freecad-e74ef5c003/tests/grader/reference.FCStd": "8094816aff087258f2361b4c0fc99c661c04d0bbf43ca8fd03c14c8408cd892a", "freecad-e74ef5c004/instruction.md": "89fc0a7b200260da7a5233d4a19b5dcdbbd13ee0ef5f875af6fc193f0d198ac6", "freecad-e74ef5c004/tests/run_scorer.py": "0ecbf3f5844136ffdb0d549a042f4356164c709d53a8a38eb139363cf5523b73", "freecad-e74ef5c004/tests/grader/spec.json": "9b3a48867c44663f86cee7c3de887bf1a0c3a62fc41e24d78319db7e7497dc95", "freecad-e74ef5c004/tests/grader/param_check.py": "871fe3d82c2281dd55d7e6f40f179ee8e6928dd7bb50cdeb0983faf4f9ac58fb", "freecad-e74ef5c004/tests/grader/reference.FCStd": "d6430c93a0a9fb5e2e8b144c3627b8c97ee0a2512b6a113b08d73d100b7b4751", "freecad-e74ef5c005/instruction.md": "6f83c7add4ddcb2bb9eb262da3d3fce349263fe2b6cceed19857c61aeb6327e1", "freecad-e74ef5c005/tests/run_scorer.py": "0ecbf3f5844136ffdb0d549a042f4356164c709d53a8a38eb139363cf5523b73", "freecad-e74ef5c005/tests/grader/spec.json": "cce7700b95b44c9171534612f1142ee5863e0b48c9d5a11fbb8c5ed5dbce37bb", "freecad-e74ef5c005/tests/grader/param_check.py": "c52757fba931b4f680e970cdf50549dc32184b97bb72c4e5e74e4d4fe95ef349", "freecad-e74ef5c005/tests/grader/reference.FCStd": "bab90ab93de3c3900615458a88a2f4ab5864440927c8f289f784c8cf969b4339", "freecad-ecdc59b900/instruction.md": "a486ed87ad96dc5f6f55a78f1359b02cc4dbf646e44d2b519138fdb568b6f05a", "freecad-ecdc59b900/tests/run_scorer.py": "0ecbf3f5844136ffdb0d549a042f4356164c709d53a8a38eb139363cf5523b73", "freecad-ecdc59b900/tests/grader/spec.json": "bb75a907fb2e6f6d210ee98ca9994b738566a26322ded063e2a1e0893048929d", "freecad-ecdc59b900/tests/grader/param_check.py": "2e15d998420e14cadb67fc639adc8798555e219375ef89d34cc94415a2c54558", "freecad-ecdc59b900/tests/grader/reference.FCStd": "e26a53af63214bf662b3a06b413d3a7bc8126aa9599871c4c4f51e6ba8792e44", "freecad-eceefa2929/instruction.md": "99f3018ae9976a13061322844a9e8afdd3aecee45c1d891090cb2b66eea22e7c", "freecad-eceefa2929/tests/run_scorer.py": "0ecbf3f5844136ffdb0d549a042f4356164c709d53a8a38eb139363cf5523b73", "freecad-eceefa2929/tests/grader/spec.json": "dc92abc9b27e44b7b088c82fc55f1a2e7ec4819a42a85f6ce5ae95db0834012c", "freecad-eceefa2929/tests/grader/param_check.py": "dc68e04a1bd730e0ffd5d119b36edeeba2d464fda12cd093f23798299ca90818", "freecad-eceefa2929/tests/grader/reference.FCStd": "4bacf340a6419428fa004f72c9fdeb672ff1e506838108ef406a7c2d31e5bf7e", "freecad-ecf3ef894f/instruction.md": "07bbd1ed708dc531acc29ff023b701cb383c2119bd63876c4d9d1fe7a15b4542", "freecad-ecf3ef894f/tests/run_scorer.py": "0ecbf3f5844136ffdb0d549a042f4356164c709d53a8a38eb139363cf5523b73", "freecad-ecf3ef894f/tests/grader/spec.json": "e90c8499c849e61aa88dfa550a417c3963a7e38721c6f04dc70ed19c3f242489", "freecad-ecf3ef894f/tests/grader/param_check.py": "a818f5d9384a7cac59587073bb063803e18b85ce14d8b53e346bce75d172ed4d", "freecad-ecf3ef894f/tests/grader/reference.FCStd": "1074f39b393d8cbf859d6af0440f821adc629a60d58601de22f9be8a8b857af1", "freecad-f0d323e9ce/instruction.md": "ecc63fe107d2de99a5cca2686de3ec542ac5489860f83324a600bbf9f50678ee", "freecad-f0d323e9ce/tests/run_scorer.py": "0ecbf3f5844136ffdb0d549a042f4356164c709d53a8a38eb139363cf5523b73", "freecad-f0d323e9ce/tests/grader/spec.json": "80b1df871523fc22c984153a766d1a70a60677fd5750872b363677d02f7e2609", "freecad-f0d323e9ce/tests/grader/param_check.py": "4b74978d4c1da9c40e3798318547f7896b81f969a6623fb0092160634bf0257a", "freecad-f0d323e9ce/tests/grader/reference.FCStd": "26db1e94e0af2115b7da1b348b55f8dbe960efa7af1f7d897cd9d51030e49bcf", "freecad-f3e10795e7/instruction.md": "77c710d9100296d117f9b219da83faf88b4936d4770c8e38d6b8d014e1a4532d", "freecad-f3e10795e7/tests/run_scorer.py": "0ecbf3f5844136ffdb0d549a042f4356164c709d53a8a38eb139363cf5523b73", "freecad-f3e10795e7/tests/grader/spec.json": "20d0c2277f3cddfa05a0995ece541ad6762cb125ac72159d24b8e727e6350d52", "freecad-f3e10795e7/tests/grader/param_check.py": "669d31345b69e6075dcce98018c9d9ba5691a9f4edbc1540bf170b70a67f7c1a", "freecad-f3e10795e7/tests/grader/reference.FCStd": "e4ce41c3789b85efd43bc2c934b32588cd23d8483741258037bfc624b6003cb7", "freecad-f67dad96d1/instruction.md": "5de7b8fb1b69d80afca964945956f05ff5c87b3e9733e18a690fc119ec983c34", "freecad-f67dad96d1/tests/run_scorer.py": "0ecbf3f5844136ffdb0d549a042f4356164c709d53a8a38eb139363cf5523b73", "freecad-f67dad96d1/tests/grader/spec.json": "af12195d0c6869e758f2aea98d2afe5fb8c2015787f1a6061cefea654ed71e39", "freecad-f67dad96d1/tests/grader/param_check.py": "ad3129b9f82c8dacf10ade0fa873c64c6f4f5e843f5df9193c8a376f4146d6f6", "freecad-f67dad96d1/tests/grader/reference.FCStd": "2c659089a6bc173f39d3729cfbd4f2d3e9e2ab89d8aa2a8799e324aa29df92fe", "freecad-fa1a50cf2b/instruction.md": "bcff1fd662cac233d2bc83f0251d4ef67e5d54ec65bd5de428aa7996562d81dd", "freecad-fa1a50cf2b/tests/run_scorer.py": "0ecbf3f5844136ffdb0d549a042f4356164c709d53a8a38eb139363cf5523b73", "freecad-fa1a50cf2b/tests/grader/spec.json": "f14d53761ea9d643852957619efc25241a2dddb4840bce2fff3ab857676d46cc", "freecad-fa1a50cf2b/tests/grader/param_check.py": "b893ecc5248be5734e9460fcf425e0b02c5771136f6bb8490ed5665293fc198f", "freecad-fa1a50cf2b/tests/grader/reference.FCStd": "63407f7e0f2acef291049cc4e3eda60f019cf59deb150649ca1e4069e9446dc9", "freecad-fc1e98cd09/instruction.md": "d008cf44486b78a3b80a61966204c6824703555956287248315d42e6bf63883c", "freecad-fc1e98cd09/tests/run_scorer.py": "0ecbf3f5844136ffdb0d549a042f4356164c709d53a8a38eb139363cf5523b73", "freecad-fc1e98cd09/tests/grader/spec.json": "6c0fb2b160ba801b9db8b808fc0b8478c73b0dc66dd0e0359817775a442503de", "freecad-fc1e98cd09/tests/grader/param_check.py": "85dff2f995733e5e9778983fbbb92625e88a5a5297b47920ec647874992c3ea0", "freecad-fc1e98cd09/tests/grader/reference.FCStd": "4a03e38feac15c57742abb391f3b1b2353ed3276a5669dca98112f0c4dadc6c8"}''')
bad = [k for k, h in HASHES.items() if not (TASKS / k).exists() or hashlib.sha256((TASKS / k).read_bytes()).hexdigest() != h]
assert not bad, f'{len(bad)} task files differ from the verified suite, e.g. {bad[:3]}'
print(len(list(TASKS.glob('freecad-*'))), 'tasks;', len(HASHES), 'files verified')
Path('/content/cadbench_colab_agent.py').write_bytes(base64.b64decode('''IiIiT3BlbiBtb2RlbHMgb24gUGFyYW1ldHJpYyBDQUQgQmVuY2ggdjIgd2l0aCBhbiBleGVjdXRpb24tZmVlZGJhY2sgcmVwYWlyIGxvb3AgKHJ1bnMgb24gYSBDb2xhYiBHUFUpLgoKUm91bmQgMCBzZW5kcyBlYWNoIHRhc2sncyB2ZXJiYXRpbSBIYXJib3IgaW5zdHJ1Y3Rpb24ubWQgcGx1cyBvbmUgbGluZSBhc2tpbmcgZm9yIHRoZSBjb21wbGV0ZSBhbnN3ZXIucHkuCkV2ZXJ5IHNjcmlwdCBpcyBleGVjdXRlZCBieSBjb25kYS1mb3JnZSBGcmVlQ0FEIDEuMS4wICh0aGUgb2ZmaWNpYWwgdmVyaWZpZXIgcnVudGltZSkuIEEgZmFpbGVkIGF0dGVtcHQgLS0KYSBQeXRob24gZXJyb3IsIG5vIGFuc3dlci5GQ1N0ZCBuZXh0IHRvIHRoZSBzY3JpcHQsIG9yIGEgZG9jdW1lbnQgdGhhdCB2aW9sYXRlcyB0aGUgaW5zdHJ1Y3Rpb24ncyBzaW5nbGUKUGFydERlc2lnbiBCb2R5IC8gZWRpdGFibGUgZmVhdHVyZS10cmVlIHJlcXVpcmVtZW50ICh0aGUgcHVibGljIHZhbGlkYXRvcidzIGludGVncml0eSBnYXRlcywgd2hpY2ggcmVhZCBvbmx5CnRoZSBjYW5kaWRhdGUpIC0tIGlzIHJldHVybmVkIHRvIHRoZSBtb2RlbCB3aXRoIHRoZSBlcnJvciB0ZXh0IGZvciB1cCB0byAtLXJlcGFpcnMgZnVydGhlciByb3VuZHMuIE5laXRoZXIKdGhlIHJlZmVyZW5jZSBwYXJ0IG5vciB0aGUgZ3JhZGVyJ3Mgc3BlYy9wYXJhbV9jaGVjayBmaWxlcyBhcmUgdXNlZCBiZWZvcmUgZmluYWwgc2NvcmluZy4gR3JlZWR5IGRlY29kaW5nLgpUaGUg
bGl0ZXJhbCBIYXJib3IgcGF0aCAvYXBwL2Fuc3dlci5GQ1N0ZCBpcyBtYXBwZWQgdG8gdGhlIHRhc2sgZm9sZGVyLCB3aGVyZSAvYXBwIGRvZXMgbm90IGV4aXN0LgoKICBydW4gICA6IGdlbmVyYXRlICsgZXhlY3V0ZSArIHJlcGFpciBmb3Igb25lIG1vZGVsLCBhbGwgdGFza3MgYmF0Y2hlZCBwZXIgcm91bmQgdGhyb3VnaCB2TExNCiAgc2NvcmUgOiBzY29yZSB0aGUgZmluYWwgYW5zd2VycyB3aXRoIGVhY2ggdGFzaydzIHRlc3RzL3J1bl9zY29yZXIucHkgKGdudWNsZXVzLWZyZWVjYWQtdmFsaWRhdG9yIDAuNC4wKQoiIiIKZnJvbSBfX2Z1dHVyZV9fIGltcG9ydCBhbm5vdGF0aW9ucwoKaW1wb3J0IGFyZ3BhcnNlCmZyb20gY29uY3VycmVudC5mdXR1cmVzIGltcG9ydCBUaHJlYWRQb29sRXhlY3V0b3IKaW1wb3J0IGpzb24KaW1wb3J0IG9zCmZyb20gcGF0aGxpYiBpbXBvcnQgUGF0aAppbXBvcnQgcmUKaW1wb3J0IHN1YnByb2Nlc3MKaW1wb3J0IHRpbWUKCkFTSyA9ICJcblxuUmVwbHkgd2l0aCB0aGUgY29tcGxldGUgY29udGVudHMgb2YgYW5zd2VyLnB5IGluIGEgc2luZ2xlIGBgYHB5dGhvbiBjb2RlIGJsb2NrIGFuZCBub3RoaW5nIGVsc2UuIgpGSVggPSAiRml4IHRoZSBzY3JpcHQuIFJlcGx5IHdpdGggdGhlIGNvbXBsZXRlIGNvcnJlY3RlZCBhbnN3ZXIucHkgaW4gYSBzaW5nbGUgYGBgcHl0aG9uIGNvZGUgYmxvY2sgYW5kIG5vdGhpbmcgZWxzZS4iCkdBVEUgPSAnJydpbXBvcnQganNvbiwgc3lzCmltcG9y
dCBGcmVlQ0FECmZyb20gZnJlZWNhZF92YWxpZGF0b3IuY29tcGFyYXRvcnMuaW50ZWdyaXR5X2dhdGVzIGltcG9ydCBwYXJ0ZGVzaWduX2JvZHlfZ2F0ZSwgcGFydGRlc2lnbl9mZWF0dXJlX3RyZWVfZ2F0ZQpkb2MgPSBGcmVlQ0FELm9wZW5Eb2N1bWVudChzeXMuYXJndlsxXSkKdHJ5OgogICAgcHJpbnQoanNvbi5kdW1wcyh7InJlYXNvbiI6IHBhcnRkZXNpZ25fYm9keV9nYXRlKGRvYykgb3IgcGFydGRlc2lnbl9mZWF0dXJlX3RyZWVfZ2F0ZShkb2MpfSkpCmZpbmFsbHk6CiAgICBGcmVlQ0FELmNsb3NlRG9jdW1lbnQoZG9jLk5hbWUpCicnJwoKCmRlZiBleHRyYWN0X2NvZGUodGV4dDogc3RyKSAtPiBzdHIgfCBOb25lOgogICAgYmxvY2tzID0gcmUuZmluZGFsbChyImBgYCg/OnB5dGhvbnxweSk/XHMqXG4oLio/KWBgYCIsIHRleHQsIHJlLlMpCiAgICBpZiBibG9ja3M6CiAgICAgICAgcmV0dXJuIG1heChibG9ja3MsIGtleT1sZW4pCiAgICBvcGVuZWQgPSByZS5zZWFyY2gociJgYGAoPzpweXRob258cHkpP1xzKlxuKC4qKSIsIHRleHQsIHJlLlMpICAjIHVuY2xvc2VkIGZlbmNlLCBlLmcuIGEgcmVwbHkgY3V0IGF0IHRoZSBjYXAKICAgIGlmIG9wZW5lZDoKICAgICAgICByZXR1cm4gb3BlbmVkLmdyb3VwKDEpCiAgICByZXR1cm4gdGV4dCBpZiAiaW1wb3J0IEZyZWVDQUQiIGluIHRleHQgZWxzZSBOb25lCgoKZGVmIGV4ZWN1dGUoZm9sZGVyOiBQYXRoLCBjb2RlOiBzdHIsIGZjX3B5dGhvbjogc3RyLCBlbnY6
IGRpY3QpIC0+IGRpY3Q6CiAgICAiIiJSdW4gb25lIGFuc3dlci5weSB3aXRoIEZyZWVDQUQ7IHJldHVybiB7b2ssIGZlZWRiYWNrLCBzdGF0dXN9LiIiIgogICAgc2NyaXB0LCBmY3N0ZCA9IGZvbGRlciAvICJhbnN3ZXIucHkiLCBmb2xkZXIgLyAiYW5zd2VyLkZDU3RkIgogICAgZm9yIHN0YWxlIGluIChmY3N0ZCwgZm9sZGVyIC8gIl9ydW4uRkNTdGQiKToKICAgICAgICBzdGFsZS51bmxpbmsobWlzc2luZ19vaz1UcnVlKQogICAgc2NyaXB0LndyaXRlX3RleHQoY29kZSkKICAgIHJ1bm5hYmxlID0gZm9sZGVyIC8gIl9ydW4ucHkiCiAgICBydW5uYWJsZS53cml0ZV90ZXh0KGNvZGUucmVwbGFjZSgiL2FwcC9hbnN3ZXIuRkNTdGQiLCBzdHIoZmNzdGQpKS5yZXBsYWNlKCIvYXBwL2Fuc3dlci5weSIsIHN0cihzY3JpcHQpKSkKICAgIHRyeToKICAgICAgICBwcm9jID0gc3VicHJvY2Vzcy5ydW4oW2ZjX3B5dGhvbiwgcnVubmFibGUubmFtZV0sIGN3ZD1mb2xkZXIsIGNhcHR1cmVfb3V0cHV0PVRydWUsIHRleHQ9VHJ1ZSwgdGltZW91dD0xMjAsIGVudj1lbnYpCiAgICBleGNlcHQgc3VicHJvY2Vzcy5UaW1lb3V0RXhwaXJlZDoKICAgICAgICByZXR1cm4gZGljdChvaz1GYWxzZSwgc3RhdHVzPSJ0aW1lb3V0IiwgZmVlZGJhY2s9IlJ1bm5pbmcgYW5zd2VyLnB5IHdpdGggRnJlZUNBRCAxLjEuMCBkaWQgbm90IGZpbmlzaCB3aXRoaW4gIgogICAgICAgICAgICAgICAgICAgICIxMjAgc2Vjb25kcy4iKQogICAgaWYgKGZv
bGRlciAvICJfcnVuLkZDU3RkIikuZXhpc3RzKCkgYW5kIG5vdCBmY3N0ZC5leGlzdHMoKToKICAgICAgICAoZm9sZGVyIC8gIl9ydW4uRkNTdGQiKS5yZW5hbWUoZmNzdGQpCiAgICBpZiBwcm9jLnJldHVybmNvZGUgIT0gMDoKICAgICAgICB0YWlsID0gKHByb2Muc3RkZXJyIG9yIHByb2Muc3Rkb3V0KVstMjUwMDpdCiAgICAgICAgcmV0dXJuIGRpY3Qob2s9RmFsc2UsIHN0YXR1cz0ic2NyaXB0X2Vycm9yIiwKICAgICAgICAgICAgICAgICAgICBmZWVkYmFjaz1mIkkgcmFuIGFuc3dlci5weSB3aXRoIEZyZWVDQUQgMS4xLjAgYW5kIGl0IGZhaWxlZDpcbmBgYFxue3RhaWx9XG5gYGAiKQogICAgaWYgbm90IGZjc3RkLmV4aXN0cygpOgogICAgICAgIHJldHVybiBkaWN0KG9rPUZhbHNlLCBzdGF0dXM9Im5vX2Zjc3RkIiwgZmVlZGJhY2s9ImFuc3dlci5weSByYW4sIGJ1dCBubyBhbnN3ZXIuRkNTdGQgd2FzIHdyaXR0ZW4gbmV4dCB0byB0aGUgIgogICAgICAgICAgICAgICAgICAgICJzY3JpcHQuIERlcml2ZSB0aGUgb3V0cHV0IHBhdGggZnJvbSBfX2ZpbGVfXyBhbmQgc2F2ZSB0aGUgZG9jdW1lbnQgdGhlcmUuIikKICAgIGdhdGUgPSBmb2xkZXIgLyAiX2dhdGUucHkiCiAgICBnYXRlLndyaXRlX3RleHQoR0FURSkKICAgIHByb2MgPSBzdWJwcm9jZXNzLnJ1bihbZmNfcHl0aG9uLCBnYXRlLm5hbWUsIHN0cihmY3N0ZCldLCBjd2Q9Zm9sZGVyLCBjYXB0dXJlX291dHB1dD1UcnVlLCB0ZXh0PVRydWUsIHRpbWVvdXQ9
MTIwLAogICAgICAgICAgICAgICAgICAgICAgICAgIGVudj1lbnYpCiAgICB0cnk6CiAgICAgICAgcmVhc29uID0ganNvbi5sb2Fkcyhwcm9jLnN0ZG91dC5zdHJpcCgpLnNwbGl0bGluZXMoKVstMV0pWyJyZWFzb24iXQogICAgZXhjZXB0IEV4Y2VwdGlvbjogICMgbm9xYTogQkxFMDAxIC0gYW4gdW5yZWFkYWJsZSBkb2N1bWVudCBpcyBhIGZhaWx1cmUKICAgICAgICByZWFzb24gPSAidGhlIHNhdmVkIGFuc3dlci5GQ1N0ZCBjb3VsZCBub3QgYmUgb3BlbmVkOiAiICsgcHJvYy5zdGRlcnJbLTgwMDpdCiAgICBpZiByZWFzb246CiAgICAgICAgcmV0dXJuIGRpY3Qob2s9RmFsc2UsIHN0YXR1cz0iZ2F0ZV9mYWlsZWQiLCBmZWVkYmFjaz1mImFuc3dlci5weSByYW4sIGJ1dCB0aGUgc2F2ZWQgbW9kZWwgdmlvbGF0ZXMgdGhlIHRhc2sgIgogICAgICAgICAgICAgICAgICAgIGYicmVxdWlyZW1lbnRzOiB7cmVhc29ufSIpCiAgICByZXR1cm4gZGljdChvaz1UcnVlLCBzdGF0dXM9Im9rIiwgZmVlZGJhY2s9Tm9uZSkKCgpkZWYgcnVuKHRhc2tzOiBQYXRoLCBvdXQ6IFBhdGgsIG1vZGVsOiBzdHIsIHRhZzogc3RyLCByZXBhaXJzOiBpbnQsIGZjX3B5dGhvbjogc3RyLCB3b3JrZXJzOiBpbnQsCiAgICAgICAgbWF4X21vZGVsX2xlbjogaW50LCBtYXhfdG9rZW5zOiBpbnQsIGdwdV91dGlsOiBmbG9hdCkgLT4gTm9uZToKICAgIGZyb20gdHJhbnNmb3JtZXJzIGltcG9ydCBBdXRvVG9rZW5pemVyCiAgICBmcm9tIHZsbG0gaW1w
b3J0IExMTSwgU2FtcGxpbmdQYXJhbXMKCiAgICBlbnYgPSBkaWN0KG9zLmVudmlyb24pCiAgICBpZHMgPSBzb3J0ZWQocC5uYW1lLnJlbW92ZXByZWZpeCgiZnJlZWNhZC0iKSBmb3IgcCBpbiB0YXNrcy5nbG9iKCJmcmVlY2FkLSoiKSkKICAgIHJvb3QgPSBvdXQgLyB0YWcKICAgIHJvb3QubWtkaXIocGFyZW50cz1UcnVlLCBleGlzdF9vaz1UcnVlKQogICAgKHJvb3QgLyAicHJvdG9jb2wuanNvbiIpLndyaXRlX3RleHQoanNvbi5kdW1wcyhkaWN0KG1vZGVsPW1vZGVsLCByZXBhaXJzPXJlcGFpcnMsIGRlY29kaW5nPSJncmVlZHkiLAogICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgIG1heF90b2tlbnM9bWF4X3Rva2VucywgbWF4X21vZGVsX2xlbj1tYXhfbW9kZWxfbGVuLAogICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgIGFzaz1BU0ssIGZpeD1GSVgsIHRhc2tzPWxlbihpZHMpKSwgaW5kZW50PTIpICsgIlxuIikKICAgIHRva2VuaXplciA9IEF1dG9Ub2tlbml6ZXIuZnJvbV9wcmV0cmFpbmVkKG1vZGVsKQogICAgbGxtID0gTExNKG1vZGVsPW1vZGVsLCBtYXhfbW9kZWxfbGVuPW1heF9tb2RlbF9sZW4sIGdwdV9tZW1vcnlfdXRpbGl6YXRpb249Z3B1X3V0aWwsIHNlZWQ9MCkKICAgIHBhcmFtcyA9IFNhbXBsaW5nUGFyYW1zKHRlbXBlcmF0dXJlPTAuMCwgbWF4X3Rva2Vucz1tYXhfdG9rZW5zKQogICAgc3RhdGUg
PSB7dGlkOiBkaWN0KGluc3RydWN0aW9uPSh0YXNrcyAvIGYiZnJlZWNhZC17dGlkfSIgLyAiaW5zdHJ1Y3Rpb24ubWQiKS5yZWFkX3RleHQoKSwgaGlzdG9yeT1bXSwKICAgICAgICAgICAgICAgICAgICAgICBkb25lPUZhbHNlKSBmb3IgdGlkIGluIGlkc30KICAgIHJvdW5kcyA9IFtdCiAgICBmb3IgciBpbiByYW5nZShyZXBhaXJzICsgMSk6CiAgICAgICAgYWN0aXZlID0gW3RpZCBmb3IgdGlkIGluIGlkcyBpZiBub3Qgc3RhdGVbdGlkXVsiZG9uZSJdXQogICAgICAgIGlmIG5vdCBhY3RpdmU6CiAgICAgICAgICAgIGJyZWFrCiAgICAgICAgcHJvbXB0cyA9IFtdCiAgICAgICAgZm9yIHRpZCBpbiBhY3RpdmU6CiAgICAgICAgICAgIHMgPSBzdGF0ZVt0aWRdCiAgICAgICAgICAgIG1lc3NhZ2VzID0gW2RpY3Qocm9sZT0idXNlciIsIGNvbnRlbnQ9c1siaW5zdHJ1Y3Rpb24iXSArIEFTSyldCiAgICAgICAgICAgIGlmIHNbImhpc3RvcnkiXTogICMgY29tcGFjdCByZXBhaXIgY29udGV4dDogb3JpZ2luYWwgdGFzaywgbGFzdCBzY3JpcHQsIGxhc3QgZmVlZGJhY2sKICAgICAgICAgICAgICAgIGxhc3QgPSBzWyJoaXN0b3J5Il1bLTFdCiAgICAgICAgICAgICAgICBtZXNzYWdlcyArPSBbZGljdChyb2xlPSJhc3Npc3RhbnQiLCBjb250ZW50PWxhc3RbInJlcGx5Il0pLAogICAgICAgICAgICAgICAgICAgICAgICAgICAgIGRpY3Qocm9sZT0idXNlciIsIGNvbnRlbnQ9bGFzdFsiZmVlZGJhY2siXSArICJcblxuIiArIEZJ
WCldCiAgICAgICAgICAgIHByb21wdCA9IHRva2VuaXplci5hcHBseV9jaGF0X3RlbXBsYXRlKG1lc3NhZ2VzLCB0b2tlbml6ZT1GYWxzZSwgYWRkX2dlbmVyYXRpb25fcHJvbXB0PVRydWUpCiAgICAgICAgICAgIGlmIGxlbih0b2tlbml6ZXIocHJvbXB0KS5pbnB1dF9pZHMpICsgbWF4X3Rva2VucyA+IG1heF9tb2RlbF9sZW46CiAgICAgICAgICAgICAgICBtZXNzYWdlcyA9IG1lc3NhZ2VzWzoxXQogICAgICAgICAgICAgICAgcHJvbXB0ID0gdG9rZW5pemVyLmFwcGx5X2NoYXRfdGVtcGxhdGUobWVzc2FnZXMsIHRva2VuaXplPUZhbHNlLCBhZGRfZ2VuZXJhdGlvbl9wcm9tcHQ9VHJ1ZSkKICAgICAgICAgICAgcHJvbXB0cy5hcHBlbmQocHJvbXB0KQogICAgICAgIHN0YXJ0ZWQgPSB0aW1lLnBlcmZfY291bnRlcigpCiAgICAgICAgb3V0cHV0cyA9IFtvLm91dHB1dHNbMF0gZm9yIG8gaW4gbGxtLmdlbmVyYXRlKHByb21wdHMsIHBhcmFtcyldCiAgICAgICAgcmVwbGllcyA9IFtvLnRleHQgZm9yIG8gaW4gb3V0cHV0c10KICAgICAgICB0cnVuY2F0ZWQgPSBbby5maW5pc2hfcmVhc29uID09ICJsZW5ndGgiIGZvciBvIGluIG91dHB1dHNdCiAgICAgICAgZ2VuX3NlY29uZHMgPSB0aW1lLnBlcmZfY291bnRlcigpIC0gc3RhcnRlZAoKICAgICAgICBkZWYgYXR0ZW1wdChpdGVtKToKICAgICAgICAgICAgdGlkLCByZXBseSwgY3V0ID0gaXRlbQogICAgICAgICAgICBmb2xkZXIgPSByb290IC8gdGlkCiAgICAgICAgICAg
IGZvbGRlci5ta2RpcihleGlzdF9vaz1UcnVlKQogICAgICAgICAgICBjb2RlID0gZXh0cmFjdF9jb2RlKHJlcGx5KQogICAgICAgICAgICByZXN1bHQgPSBleGVjdXRlKGZvbGRlciwgY29kZSwgZmNfcHl0aG9uLCBlbnYpIGlmIGNvZGUgZWxzZSBkaWN0KAogICAgICAgICAgICAgICAgb2s9RmFsc2UsIHN0YXR1cz0ibm9fY29kZSIsIGZlZWRiYWNrPSJZb3VyIHJlcGx5IGNvbnRhaW5lZCBubyBweXRob24gY29kZSBibG9jay4iKQogICAgICAgICAgICBpZiBjdXQgYW5kIG5vdCByZXN1bHRbIm9rIl06CiAgICAgICAgICAgICAgICByZXN1bHRbImZlZWRiYWNrIl0gPSAoZiJZb3VyIHJlcGx5IHdhcyBjdXQgb2ZmIGF0IHRoZSB7bWF4X3Rva2Vuc30tdG9rZW4gbGltaXQsIHNvIGFuc3dlci5weSBpcyAiCiAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgImluY29tcGxldGU7IHdyaXRlIGEgc2hvcnRlciBzY3JpcHQgd2l0aG91dCByZXBlYXRlZCBsaW5lcy5cbiIgKyByZXN1bHRbImZlZWRiYWNrIl0pCiAgICAgICAgICAgIHJldHVybiB0aWQsIHJlcGx5LCBjdXQsIHJlc3VsdAogICAgICAgIHdpdGggVGhyZWFkUG9vbEV4ZWN1dG9yKG1heF93b3JrZXJzPXdvcmtlcnMpIGFzIHBvb2w6CiAgICAgICAgICAgIGZvciB0aWQsIHJlcGx5LCBjdXQsIHJlc3VsdCBpbiBwb29sLm1hcChhdHRlbXB0LCB6aXAoYWN0aXZlLCByZXBsaWVzLCB0cnVuY2F0ZWQpKToKICAgICAgICAgICAgICAgIHMgPSBzdGF0ZVt0
aWRdCiAgICAgICAgICAgICAgICBzWyJoaXN0b3J5Il0uYXBwZW5kKGRpY3Qocm91bmQ9ciwgcmVwbHk9cmVwbHksIHRydW5jYXRlZD1jdXQsIHN0YXR1cz1yZXN1bHRbInN0YXR1cyJdLAogICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgIGZlZWRiYWNrPXJlc3VsdFsiZmVlZGJhY2siXSkpCiAgICAgICAgICAgICAgICBzWyJkb25lIl0gPSByZXN1bHRbIm9rIl0KICAgICAgICBvayA9IHN1bShzdGF0ZVt0XVsiZG9uZSJdIGZvciB0IGluIGlkcykKICAgICAgICByb3VuZHMuYXBwZW5kKGRpY3Qocm91bmQ9ciwgYXR0ZW1wdGVkPWxlbihhY3RpdmUpLCBwYXNzaW5nX3RvdGFsPW9rLCBnZW5fc2Vjb25kcz1yb3VuZChnZW5fc2Vjb25kcywgMSkpKQogICAgICAgIHByaW50KGYie3RhZ30gcm91bmQge3J9OiBhdHRlbXB0ZWQge2xlbihhY3RpdmUpfSwgcGFzc2luZyBleGVjdXRpb24rZ2F0ZXMge29rfS97bGVuKGlkcyl9LCAiCiAgICAgICAgICAgICAgZiJnZW5lcmF0aW9uIHtnZW5fc2Vjb25kczouMGZ9cyIsIGZsdXNoPVRydWUpCiAgICBmb3IgdGlkLCBzIGluIHN0YXRlLml0ZW1zKCk6CiAgICAgICAgZm9sZGVyID0gcm9vdCAvIHRpZAogICAgICAgIGZvbGRlci5ta2RpcihleGlzdF9vaz1UcnVlKQogICAgICAgIChmb2xkZXIgLyAidHJhbnNjcmlwdC5qc29uIikud3JpdGVfdGV4dChqc29uLmR1bXBzKGRpY3QocGFzc2VkPXNbImRvbmUiXSwgYXR0ZW1wdHM9c1siaGlzdG9yeSJdKSwgaW5kZW50
PTIpKQogICAgKHJvb3QgLyAicm91bmRzLmpzb24iKS53cml0ZV90ZXh0KGpzb24uZHVtcHMocm91bmRzLCBpbmRlbnQ9MikgKyAiXG4iKQoKCmRlZiBzY29yZSh0YXNrczogUGF0aCwgb3V0OiBQYXRoLCB0YWc6IHN0ciwgZmNfcHl0aG9uOiBzdHIsIHdvcmtlcnM6IGludCkgLT4gZGljdDoKICAgIHJvb3QgPSBvdXQgLyB0YWcKICAgIGlkcyA9IHNvcnRlZChwLm5hbWUucmVtb3ZlcHJlZml4KCJmcmVlY2FkLSIpIGZvciBwIGluIHRhc2tzLmdsb2IoImZyZWVjYWQtKiIpKQogICAgZW52ID0gZGljdChvcy5lbnZpcm9uKQoKICAgIGRlZiBvbmUodGlkKToKICAgICAgICB0YXNrLCBmb2xkZXIgPSB0YXNrcyAvIGYiZnJlZWNhZC17dGlkfSIsIHJvb3QgLyB0aWQKICAgICAgICByb3cgPSBkaWN0KHRhc2s9dGlkLCBjb21iaW5lZD0wLjAsIGdlb21ldHJ5PTAuMCwgc3BlYz0wLjAsIHN0YXR1cz0ibWlzc2luZ19jYW5kaWRhdGUiKQogICAgICAgIGlmIChmb2xkZXIgLyAiYW5zd2VyLkZDU3RkIikuZXhpc3RzKCk6CiAgICAgICAgICAgIHByb2MgPSBzdWJwcm9jZXNzLnJ1bihbZmNfcHl0aG9uLCBzdHIodGFzayAvICJ0ZXN0cy9ydW5fc2NvcmVyLnB5IiksICItLXJlZmVyZW5jZSIsCiAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgc3RyKHRhc2sgLyAidGVzdHMvZ3JhZGVyL3JlZmVyZW5jZS5GQ1N0ZCIpLCAiLS1jYW5kaWRhdGUiLCBzdHIoZm9sZGVyIC8gImFuc3dlci5GQ1N0ZCIpLAogICAgICAgICAgICAg
ICAgICAgICAgICAgICAgICAgICAgICItLXNwZWMiLCBzdHIodGFzayAvICJ0ZXN0cy9ncmFkZXIvc3BlYy5qc29uIiksICItLXJld2FyZC10eHQiLAogICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgIHN0cihmb2xkZXIgLyAicmV3YXJkLnR4dCIpLCAiLS1yZXdhcmQtanNvbiIsIHN0cihmb2xkZXIgLyAicmV3YXJkLmpzb24iKV0sCiAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICBjYXB0dXJlX291dHB1dD1UcnVlLCB0ZXh0PVRydWUsIHRpbWVvdXQ9NjAwLCBlbnY9ZW52KQogICAgICAgICAgICBkZXRhaWxzID0gZm9sZGVyIC8gInJld2FyZF9kZXRhaWxzLmpzb24iCiAgICAgICAgICAgIGlmIHByb2MucmV0dXJuY29kZSA9PSAwIGFuZCBkZXRhaWxzLmV4aXN0cygpOgogICAgICAgICAgICAgICAgZCA9IGpzb24ubG9hZHMoZGV0YWlscy5yZWFkX3RleHQoKSkKICAgICAgICAgICAgICAgIHJvdy51cGRhdGUoY29tYmluZWQ9ZFsiY29tYmluZWQiXSwgZ2VvbWV0cnk9ZFsiZ2VvbWV0cnlfc2ltaWxhcml0eSJdLAogICAgICAgICAgICAgICAgICAgICAgICAgICBzcGVjPWRbImNhZF9zcGVjX2NvbnNpc3RlbmN5Il0sIHN0YXR1cz0ic2NvcmVkIikKICAgICAgICAgICAgZWxzZToKICAgICAgICAgICAgICAgIHJvdy51cGRhdGUoc3RhdHVzPSJzY29yZXJfZXJyb3IiLCBlcnJvcj1wcm9jLnN0ZGVyclstNDAwOl0pCiAgICAgICAgcmV0dXJuIHJvdwogICAgd2l0aCBUaHJlYWRQb29sRXhlY3V0b3Io
bWF4X3dvcmtlcnM9d29ya2VycykgYXMgcG9vbDoKICAgICAgICByb3dzID0gbGlzdChwb29sLm1hcChvbmUsIGlkcykpCiAgICBuID0gbGVuKHJvd3MpCiAgICBzdW1tYXJ5ID0gZGljdCh0YWc9dGFnLCB0YXNrcz1uLCBzY29yZWQ9c3VtKHJbInN0YXR1cyJdID09ICJzY29yZWQiIGZvciByIGluIHJvd3MpLAogICAgICAgICAgICAgICAgICAgbWVhbl9jb21iaW5lZD1zdW0oclsiY29tYmluZWQiXSBmb3IgciBpbiByb3dzKSAvIG4sIG1lYW5fZ2VvbWV0cnk9c3VtKHJbImdlb21ldHJ5Il0gZm9yIHIgaW4gcm93cykgLyBuLAogICAgICAgICAgICAgICAgICAgbWVhbl9zcGVjPXN1bShyWyJzcGVjIl0gZm9yIHIgaW4gcm93cykgLyBuLCBwZXJfdGFzaz17clsidGFzayJdOiByIGZvciByIGluIHJvd3N9LAogICAgICAgICAgICAgICAgICAgcm91bmRzPWpzb24ubG9hZHMoKHJvb3QgLyAicm91bmRzLmpzb24iKS5yZWFkX3RleHQoKSkpCiAgICAocm9vdCAvICJzdW1tYXJ5Lmpzb24iKS53cml0ZV90ZXh0KGpzb24uZHVtcHMoc3VtbWFyeSwgaW5kZW50PTIpICsgIlxuIikKICAgIHByaW50KGpzb24uZHVtcHMoe2s6IHYgZm9yIGssIHYgaW4gc3VtbWFyeS5pdGVtcygpIGlmIGsgIT0gInBlcl90YXNrIn0sIGluZGVudD0xKSkKICAgIHJldHVybiBzdW1tYXJ5CgoKZGVmIG1haW4oKSAtPiBOb25lOgogICAgcCA9IGFyZ3BhcnNlLkFyZ3VtZW50UGFyc2VyKGRlc2NyaXB0aW9uPV9fZG9jX18sIGZvcm1hdHRlcl9jbGFzcz1h
cmdwYXJzZS5SYXdEZXNjcmlwdGlvbkhlbHBGb3JtYXR0ZXIpCiAgICBzdWIgPSBwLmFkZF9zdWJwYXJzZXJzKGRlc3Q9ImNtZCIsIHJlcXVpcmVkPVRydWUpCiAgICByID0gc3ViLmFkZF9wYXJzZXIoInJ1biIpCiAgICBzID0gc3ViLmFkZF9wYXJzZXIoInNjb3JlIikKICAgIGZvciBxIGluIChyLCBzKToKICAgICAgICBxLmFkZF9hcmd1bWVudCgiLS10YXNrcyIsIHR5cGU9UGF0aCwgcmVxdWlyZWQ9VHJ1ZSkKICAgICAgICBxLmFkZF9hcmd1bWVudCgiLS1vdXQiLCB0eXBlPVBhdGgsIHJlcXVpcmVkPVRydWUpCiAgICAgICAgcS5hZGRfYXJndW1lbnQoIi0tdGFnIiwgcmVxdWlyZWQ9VHJ1ZSkKICAgICAgICBxLmFkZF9hcmd1bWVudCgiLS1mYy1weXRob24iLCByZXF1aXJlZD1UcnVlKQogICAgICAgIHEuYWRkX2FyZ3VtZW50KCItLXdvcmtlcnMiLCB0eXBlPWludCwgZGVmYXVsdD04KQogICAgci5hZGRfYXJndW1lbnQoIi0tbW9kZWwiLCByZXF1aXJlZD1UcnVlKQogICAgci5hZGRfYXJndW1lbnQoIi0tcmVwYWlycyIsIHR5cGU9aW50LCBkZWZhdWx0PTMpCiAgICByLmFkZF9hcmd1bWVudCgiLS1tYXgtbW9kZWwtbGVuIiwgdHlwZT1pbnQsIGRlZmF1bHQ9MTYzODQpCiAgICByLmFkZF9hcmd1bWVudCgiLS1tYXgtdG9rZW5zIiwgdHlwZT1pbnQsIGRlZmF1bHQ9NDA5NikKICAgIHIuYWRkX2FyZ3VtZW50KCItLWdwdS11dGlsIiwgdHlwZT1mbG9hdCwgZGVmYXVsdD0wLjg4KQogICAgYSA9IHAucGFyc2VfYXJncygp
CiAgICBpZiBhLmNtZCA9PSAicnVuIjoKICAgICAgICBydW4oYS50YXNrcywgYS5vdXQsIGEubW9kZWwsIGEudGFnLCBhLnJlcGFpcnMsIGEuZmNfcHl0aG9uLCBhLndvcmtlcnMsIGEubWF4X21vZGVsX2xlbiwgYS5tYXhfdG9rZW5zLAogICAgICAgICAgICBhLmdwdV91dGlsKQogICAgZWxzZToKICAgICAgICBzY29yZShhLnRhc2tzLCBhLm91dCwgYS50YWcsIGEuZmNfcHl0aG9uLCBhLndvcmtlcnMpCgoKaWYgX19uYW1lX18gPT0gIl9fbWFpbl9fIjoKICAgIG1haW4oKQo='''))

In [ ]:
#@title Models
MODELS = [["qwen2.5-coder-3b", "Qwen/Qwen2.5-Coder-3B-Instruct"], ["qwen2.5-coder-7b", "Qwen/Qwen2.5-Coder-7B-Instruct"], ["qwen2.5-coder-32b-awq", "Qwen/Qwen2.5-Coder-32B-Instruct-AWQ"]]
REPAIRS = 3
if torch.cuda.mem_get_info()[1] < 30 * 2**30 and any('32b' in t for t, _ in MODELS):
    MODELS = [m for m in MODELS if '32b' not in m[0]]
    print('GPU under 30 GB: the 32B model is skipped (it needs an A100 40/80 GB)')
print(MODELS)

## Generate, execute, repair, score (background job)

Each model runs in its own process (GPU memory is released between models) and is scored right after. Re-run the wait cell after a reconnect.

In [ ]:
import subprocess, sys, time
OUT, LOG, PID = Path('/content/cadbench-out'), Path('/content/cadbench.log'), Path('/content/cadbench.pid')
def alive():
    try:
        os.kill(int(PID.read_text()), 0); return True
    except (OSError, ValueError):
        return False
if alive():
    print('Already running')
else:
    steps = []
    for tag, model in MODELS:
        if (OUT / tag / 'summary.json').exists(): continue
        common = f'--tasks {TASKS} --out {OUT} --tag {tag} --fc-python /content/fcenv/bin/python'
        steps.append(f'{sys.executable} -u /content/cadbench_colab_agent.py run {common} --model {model} --repairs {REPAIRS}')
        steps.append(f'{sys.executable} -u /content/cadbench_colab_agent.py score {common}')
    proc = subprocess.Popen('(' + ' && '.join(steps) + f') > {LOG} 2>&1; echo EXIT=$? >> {LOG}', shell=True, start_new_session=True)
    PID.write_text(str(proc.pid)); print('Started', len(steps) // 2, 'models')

In [ ]:
#@title Wait (re-run after a reconnect)
seen = 0
while True:
    lines = LOG.read_text(errors='replace').splitlines() if LOG.exists() else []
    for line in lines[seen:]:
        if any(k in line for k in ('round ', 'mean_combined', 'EXIT=', 'Error', 'Traceback')): print(line[:300], flush=True)
    seen = len(lines)
    if any(l.startswith('EXIT=') for l in lines): break
    time.sleep(60)
print('\n'.join(lines[-40:]) if 'EXIT=0' not in lines[-1] else 'All models finished')

In [ ]:
#@title Results next to the published v2 leaderboard (mean reward over 100 tasks)
PUBLISHED = [["claude-code", "anthropic/claude-fable-5-1", "max", 0.848071], ["codex", "openai/gpt-6-astra", "max", 0.847831], ["grok-build", "xai/grok-4.6", "xhigh", 0.822121], ["claude-code", "anthropic/claude-opus-5", "max", 0.795247], ["mini-swe-agent", "openrouter/moonshotai/kimi-k3", "max", 0.761764], ["claude-code", "anthropic/claude-sonnet-5", "max", 0.70524], ["codex", "openai/gpt-5.6-sol", "max", 0.703356], ["codex", "openai/gpt-5.6-terra", "max", 0.686578], ["mini-swe-agent", "openrouter/meta/muse-spark-1.3", "max", 0.654551], ["mini-swe-agent", "openrouter/z-ai/glm-5.3", "max", 0.640391]]
rows = [(f'{a} / {m} ({e})', v) for a, m, e, v in PUBLISHED]
for tag, _ in MODELS:
    s = OUT / tag / 'summary.json'
    if s.exists():
        d = json.loads(s.read_text())
        rows.append((f'{tag} + FreeCAD feedback ({REPAIRS} repairs), single-shot chat', d['mean_combined']))
        print(tag, 'passing execution+gates by round:', [r['passing_total'] for r in d['rounds']])
for name, v in sorted(rows, key=lambda r: -r[1]): print(f'{v:8.4f}  {name}')

## Save results into this notebook

Prints answer.py, transcripts and per-task scores (no FCStd; answers are re-executable) as base64 between BEGIN/END markers. **Save the notebook afterwards.**

In [ ]:
import io, tarfile
buf = io.BytesIO()
with tarfile.open(fileobj=buf, mode='w:gz') as tar:
    for f in sorted(OUT.rglob('*')):
        if f.is_file() and f.suffix in ('.py', '.json') and not f.name.startswith('_'):
            tar.add(f, arcname=str(f.relative_to(OUT)))
data = buf.getvalue(); text = base64.b64encode(data).decode()
print(f'BEGIN-RESULTS cadbench {len(data)} {hashlib.sha256(data).hexdigest()}')
for i in range(0, len(text), 1000): print(text[i:i + 1000])
print('END-RESULTS cadbench')